# API Reference

This page documents the public API of `fMRI-HA`. It is organized by module so that each function can be located from its import path, inspected through its signature, and understood through its parameters and outputs.

The sections below focus on computational, preprocessing, statistical, plotting, and pipeline-generation interfaces that can be called from Python scripts.


## Configuration

`fmriha.config`

Default configuration dictionaries for preprocessing and hyperalignment workflows.


### `get_ha_config`

**Signature**

```python
fmriha.config.get_ha_config(alignment_kind='response')
```

**Purpose**

Return a default hyperalignment pipeline config.

**Parameters**

| Parameter | Default | Meaning |
| --- | --- | --- |
| `alignment_kind` | `'response'` | Select response hyperalignment (RHA) or connectivity hyperalignment (CHA). |

**Returns or outputs**

See the function output or saved pipeline records for returned objects and generated files.

### `get_rha_config`

**Signature**

```python
fmriha.config.get_rha_config()
```

**Purpose**

Return the default response-hyperalignment pipeline config.

**Parameters**

This function takes no parameters.

**Returns or outputs**

See the function output or saved pipeline records for returned objects and generated files.

### `get_cha_config`

**Signature**

```python
fmriha.config.get_cha_config()
```

**Purpose**

Return the default connectivity-hyperalignment pipeline config.

**Parameters**

This function takes no parameters.

**Returns or outputs**

See the function output or saved pipeline records for returned objects and generated files.

### `get_preprocessing_config`

**Signature**

```python
fmriha.config.get_preprocessing_config()
```

**Purpose**

Return the default preprocessing pipeline config.

**Parameters**

This function takes no parameters.

**Returns or outputs**

See the function output or saved pipeline records for returned objects and generated files.


## Preprocessing I/O

`fmriha.preprocessing.fileio`

File loading, writing, BIDS filtering, standardization, and time-range extraction utilities.


### `path_ensure`

**Signature**

```python
fmriha.preprocessing.fileio.path_ensure(path)
```

**Purpose**

Convert a string-like path to `pathlib.Path`.

**Parameters**

| Parameter | Default | Meaning |
| --- | --- | --- |
| `path` | `required` | Input file or directory path. |

**Returns or outputs**

Path Path object pointing to the same location.

### `stand_matrix`

**Signature**

```python
fmriha.preprocessing.fileio.stand_matrix(mat, method='zscore', axis=0, ddof=1, dtype=np.float32)
```

**Purpose**

Standardize or demean a matrix along a selected axis.

**Parameters**

| Parameter | Default | Meaning |
| --- | --- | --- |
| `mat` | `required` | Input data matrix. |
| `method` | `'zscore'` | Normalization method. Use 'zscore' to subtract the mean and divide by the standard deviation, or 'demean' to subtract the mean only. |
| `axis` | `0` | Axis along which the mean and standard deviation are computed. |
| `ddof` | `1` | Degrees of freedom used by `np.nanstd`. The default is 1. |
| `dtype` | `np.float32` | Output precision. The default is `np.float32`. |

**Returns or outputs**

ndarray Normalized matrix with NaN and infinite values replaced by zero.

### `log_create`

**Signature**

```python
fmriha.preprocessing.fileio.log_create(log_filename='analysis_logging.log')
```

**Purpose**

Create or initialize a logging file.

**Parameters**

| Parameter | Default | Meaning |
| --- | --- | --- |
| `log_filename` | `'analysis_logging.log'` | Path to the log file. Parent directories are created automatically. |

**Returns or outputs**

See the function output or saved pipeline records for returned objects and generated files.

### `log_write`

**Signature**

```python
fmriha.preprocessing.fileio.log_write(log_filename, description)
```

**Purpose**

Write a message to the log file.

**Parameters**

| Parameter | Default | Meaning |
| --- | --- | --- |
| `log_filename` | `required` | Path to the log file. |
| `description` | `required` | Message or description to be written. |

**Returns or outputs**

See the function output or saved pipeline records for returned objects and generated files.

### `load_cifti_data`

**Signature**

```python
fmriha.preprocessing.fileio.load_cifti_data(cifti_file, header=False, whole=False, surface=True, medial=True, subcortical=False, dtype=np.float32)
```

**Purpose**

Load a CIFTI file and optionally extract selected data components.

**Parameters**

| Parameter | Default | Meaning |
| --- | --- | --- |
| `cifti_file` | `required` | Path to the CIFTI file. |
| `header` | `False` | If True, include the CIFTI header in the returned dictionary. |
| `whole` | `False` | If True, include the full data matrix, usually shaped as `(time_points, grayordinates)`. |
| `surface` | `True` | If True, return cortical surface data for the left and right hemispheres under `results['surface_data']`. |
| `medial` | `True` | If True, return medial wall masks for each hemisphere. Medial wall vertices are labeled as 1 and cortical vertices as 0. |
| `subcortical` | `False` | If True, return subcortical voxel data concatenated across all voxel brain models. |
| `dtype` | `np.float32` | Data precision used for the returned arrays. |

**Returns or outputs**

results : dict Dictionary containing the requested components.

### `estimate_affine_from_voxels`

**Signature**

```python
fmriha.preprocessing.fileio.estimate_affine_from_voxels(voxels, voxel_size=2.0)
```

**Purpose**

Estimate a simple affine matrix from voxel indices.

**Parameters**

| Parameter | Default | Meaning |
| --- | --- | --- |
| `voxels` | `required` | Voxel indices with shape `(n_voxels, 3)`. |
| `voxel_size` | `2.0` | Isotropic voxel size in millimeters. The default is 2.0. |

**Returns or outputs**

affine : ndarray Estimated 4 x 4 affine matrix.

### `get_cifti_subcortical`

**Signature**

```python
fmriha.preprocessing.fileio.get_cifti_subcortical(cifti_file, structure='whole', dtype=np.float32, return_img=False, return_mask=False, verbose=False)
```

**Purpose**

Extract subcortical voxel data from a CIFTI file.

**Parameters**

| Parameter | Default | Meaning |
| --- | --- | --- |
| `cifti_file` | `required` | Input CIFTI file, such as `.dtseries.nii` or `.dscalar.nii`. |
| `structure` | `'whole'` | Structure selector. Supported shortcuts are `'whole'`/`'all'`, `'L'`/`'LEFT'`, `'R'`/`'RIGHT'`, and `'BRAIN_STEM'`. A list of strings is matched against CIFTI brain structure names. |
| `dtype` | `np.float32` | Data precision used when reading CIFTI and creating the output array. |
| `return_img` | `False` | If True, return a NIfTI image instead of a raw ndarray. |
| `return_mask` | `False` | If True, also return a 3D mask indicating subcortical voxels. |
| `verbose` | `False` | If True, print selected structures. |

**Returns or outputs**

ndarray, nib.Nifti1Image, or tuple By default, returns `sub_data` with shape `(x, y, z, time)`. With `return_img=True`, returns a NIfTI image. With `return_mask=True`, also returns or directly returns a uint8 mask.

### `write_cifti`

**Signature**

```python
fmriha.preprocessing.fileio.write_cifti(reference, surface=None, subcortical=None, header=None, output_filename='output.dtseries.nii', dtype=np.float32)
```

**Purpose**

Combine surface and subcortical data, then save a CIFTI image.

**Parameters**

| Parameter | Default | Meaning |
| --- | --- | --- |
| `reference` | `required` | Reference CIFTI file used for the header and any missing data components. |
| `surface` | `None` | Surface data dictionary with keys `'l'` and `'r'`. Each value should be shaped as `(time_points, vertices)`. |
| `subcortical` | `None` | Subcortical data matrix shaped as `(time_points, voxels)`. |
| `header` | `None` | Header used for the output image. If None, the reference header is used. |
| `output_filename` | `'output.dtseries.nii'` | Path where the output CIFTI file will be saved. |
| `dtype` | `np.float32` | Data precision used when copying missing components from the reference. |

**Returns or outputs**

None

### `cifti_get_trs`

**Signature**

```python
fmriha.preprocessing.fileio.cifti_get_trs(cifti_file, TRs, output_filename, dtype=np.float32)
```

**Purpose**

Extract selected TR intervals from a CIFTI file.

**Parameters**

| Parameter | Default | Meaning |
| --- | --- | --- |
| `cifti_file` | `required` | Input CIFTI file. |
| `TRs` | `required` | One or more 1-based inclusive TR intervals. Examples: `[1, 100]` or `[[1, 100], [201, 300]]`. If None, all TRs are used. |
| `output_filename` | `required` | Output CIFTI path. If None, the extracted data matrix is returned. |
| `dtype` | `np.float32` | Data precision used for loading and returning data. |

**Returns or outputs**

ndarray or None Returns the concatenated data matrix when `output_filename` is None; otherwise saves the result and returns None.

### `cifti_combine`

**Signature**

```python
fmriha.preprocessing.fileio.cifti_combine(cifti_filelist, TRs_list=None, output_filename='out.dtseries.nii', method='demean', dtype=np.float32)
```

**Purpose**

Extract TR intervals from multiple CIFTI files and concatenate them.

**Parameters**

| Parameter | Default | Meaning |
| --- | --- | --- |
| `cifti_filelist` | `required` | CIFTI files to concatenate. |
| `TRs_list` | `None` | TR intervals for each file. If None, the full time series from every file is used. |
| `output_filename` | `'out.dtseries.nii'` | Output path ending in `.nii` or `.npy`. If None, return the concatenated data matrix. |
| `method` | `'demean'` | Optional normalization applied to each file before concatenation. |
| `dtype` | `np.float32` | Data precision used for loading and saving. |

**Returns or outputs**

ndarray or None Returns concatenated data when `output_filename` is None; otherwise saves the output and returns None.

### `load_gifti_data`

**Signature**

```python
fmriha.preprocessing.fileio.load_gifti_data(gifti_file, dtype=np.float32)
```

**Purpose**

Load data from a GIFTI file.

**Parameters**

| Parameter | Default | Meaning |
| --- | --- | --- |
| `gifti_file` | `required` | Input GIFTI file. Functional files are expected to end with `func.gii`, surface geometry files with `surf.gii`, and scalar files with `shape.gii`. |
| `dtype` | `np.float32` | Data precision used for floating-point arrays. |

**Returns or outputs**

ndarray or dict Functional data are returned as `(time_points, vertices)`. Surface geometry is returned as `{'coord': coord, 'faces': faces}`. Shape data are returned as a one-dimensional array.

### `write_gifti`

**Signature**

```python
fmriha.preprocessing.fileio.write_gifti(data, output_filename, dtype=np.float32)
```

**Purpose**

Save a time-by-vertex matrix to a functional GIFTI file.

**Parameters**

| Parameter | Default | Meaning |
| --- | --- | --- |
| `data` | `required` | Functional data with shape `(time_points, vertices)`. A 1D input is treated as a single time point. |
| `output_filename` | `required` | Output GIFTI path. The filename should end with `.gii`. |
| `dtype` | `np.float32` | Data precision used for saving. |

**Returns or outputs**

See the function output or saved pipeline records for returned objects and generated files.

### `gifti_get_trs`

**Signature**

```python
fmriha.preprocessing.fileio.gifti_get_trs(gifti_file, TRs, output_filename, dtype=np.float32)
```

**Purpose**

Extract selected TR intervals from a functional GIFTI file.

**Parameters**

| Parameter | Default | Meaning |
| --- | --- | --- |
| `gifti_file` | `required` | Input functional GIFTI file. |
| `TRs` | `required` | One or more 1-based inclusive TR intervals. Examples: `[1, 100]` or `[[1, 100], [201, 300]]`. If None, all TRs are used. |
| `output_filename` | `required` | Output GIFTI path. If None, the extracted data matrix is returned. |
| `dtype` | `np.float32` | Data precision used for loading and saving. |

**Returns or outputs**

ndarray or None Returns the concatenated data matrix when `output_filename` is None; otherwise saves the result and returns None.

### `gifti_combine`

**Signature**

```python
fmriha.preprocessing.fileio.gifti_combine(gifti_filelist, TRs_list=None, output_filename='out.func.gii', method='demean', dtype=np.float32)
```

**Purpose**

Extract TR intervals from multiple GIFTI files and concatenate them.

**Parameters**

| Parameter | Default | Meaning |
| --- | --- | --- |
| `gifti_filelist` | `required` | Functional GIFTI files to concatenate. |
| `TRs_list` | `None` | TR intervals for each file. If None, the full time series from every file is used. |
| `output_filename` | `'out.func.gii'` | Output GIFTI path. If None, return the concatenated data matrix. |
| `method` | `'demean'` | Optional normalization applied to each file before concatenation. |
| `dtype` | `np.float32` | Data precision used for loading and saving. |

**Returns or outputs**

ndarray or None Returns concatenated data when `output_filename` is None; otherwise saves the output and returns None.

### `load_nifti_data`

**Signature**

```python
fmriha.preprocessing.fileio.load_nifti_data(nifti_file, header=False, dtype=np.float32)
```

**Purpose**

Load data from a NIfTI file.

**Parameters**

| Parameter | Default | Meaning |
| --- | --- | --- |
| `nifti_file` | `required` | Path to the NIfTI file. |
| `header` | `False` | If True, include the NIfTI header and affine in the returned dictionary. |
| `dtype` | `np.float32` | Data precision used for the returned array. |

**Returns or outputs**

results : dict Dictionary containing `data` and, when requested, `header` and `affine`.

### `write_nifti`

**Signature**

```python
fmriha.preprocessing.fileio.write_nifti(data, output_filename, reference=None, dtype=np.float32)
```

**Purpose**

Save data to a NIfTI file.

**Parameters**

| Parameter | Default | Meaning |
| --- | --- | --- |
| `data` | `required` | Data matrix to save. |
| `output_filename` | `required` | Output path ending with `.nii` or `.nii.gz`. |
| `reference` | `None` | Reference image used for the affine and header. |
| `dtype` | `np.float32` | Data precision used for saving. |

**Returns or outputs**

None

### `nifti_get_trs`

**Signature**

```python
fmriha.preprocessing.fileio.nifti_get_trs(nifti_file, TRs, output_filename=None, dtype=np.float32)
```

**Purpose**

Extract specific TR ranges from a 4D NIfTI file.

**Parameters**

| Parameter | Default | Meaning |
| --- | --- | --- |
| `nifti_file` | `required` | Input 4D NIfTI file. |
| `TRs` | `required` | One or more 1-based inclusive TR intervals. Examples: `[1, 100]` or `[[1, 100], [201, 300]]`. If None, all TRs are used. |
| `output_filename` | `None` | If provided, save a new NIfTI file; otherwise return an array. |
| `dtype` | `np.float32` | Data precision used for loading and saving. |

**Returns or outputs**

ndarray or None Returns extracted data when `output_filename` is None; otherwise saves the result and returns None.

### `nifti_combine`

**Signature**

```python
fmriha.preprocessing.fileio.nifti_combine(nifti_filelist, TRs_list=None, output_filename='out_merged.nii.gz', method='demean', dtype=np.float32)
```

**Purpose**

Extract TR intervals from multiple NIfTI files and concatenate them.

**Parameters**

| Parameter | Default | Meaning |
| --- | --- | --- |
| `nifti_filelist` | `required` | Input 4D NIfTI files. |
| `TRs_list` | `None` | TR intervals for each file. If None, the full time series from every file is used. |
| `output_filename` | `'out_merged.nii.gz'` | Output file ending with `.nii`, `.nii.gz`, or `.npy`. If None, return the combined array. |
| `method` | `'demean'` | Optional standardization applied per voxel across time. |
| `dtype` | `np.float32` | Data precision used for loading and saving. |

**Returns or outputs**

ndarray or None Returns combined 4D data when `output_filename` is None; otherwise saves the output and returns None.

### `nifti_resample`

**Signature**

```python
fmriha.preprocessing.fileio.nifti_resample(src, ref_filename, output_filename=None, interpolation='nearest', vol_size=None, mode='ref', return_img=False)
```

**Purpose**

Resample a NIfTI image to a reference image or isotropic voxel size.

**Parameters**

| Parameter | Default | Meaning |
| --- | --- | --- |
| `src` | `required` | Source image to be resampled. |
| `ref_filename` | `required` | Reference NIfTI image used when `mode='ref'`. |
| `output_filename` | `None` | Optional output filename used when saving a generated file. |
| `interpolation` | `'nearest'` | Interpolation mode passed to nilearn. Common choices are `'nearest'` for masks/atlases and `'continuous'` or `'linear'` for continuous-valued images. |
| `vol_size` | `None` | Isotropic voxel size used when `mode='vox'`. |
| `mode` | `'ref'` | `'ref'` resamples to `ref_filename`. `'vox'` resamples to a new isotropic voxel size. |
| `return_img` | `False` | If True, return the resampled image instead of saving it. |

**Returns or outputs**

nibabel image or None Returns the image when `return_img=True`; otherwise saves it and returns None.

### `medial_patch`

**Signature**

```python
fmriha.preprocessing.fileio.medial_patch(surface_data, medial_wall, dtype=np.float32)
```

**Purpose**

Reinsert medial wall vertices into surface data.

**Parameters**

| Parameter | Default | Meaning |
| --- | --- | --- |
| `surface_data` | `required` | Surface data for each hemisphere. Each value must have shape `(time_points, cortical_vertices)`. |
| `medial_wall` | `required` | Medial wall masks for matching hemispheres. Values should contain 1 for medial wall vertices and 0 for cortical vertices. |
| `dtype` | `np.float32` | Data precision used for the returned arrays. |

**Returns or outputs**

data_all : dict Surface matrices with medial wall vertices filled with zeros.

### `parse_bids`

**Signature**

```python
fmriha.preprocessing.fileio.parse_bids(file_name)
```

**Purpose**

Parse BIDS-style key-value fields from a filename.

**Parameters**

| Parameter | Default | Meaning |
| --- | --- | --- |
| `file_name` | `required` | BIDS-style filename or path. |

**Returns or outputs**

name_dict : dict Parsed key-value fields. The combined suffix is stored under `'file_type'`.

### `bids_filter`

**Signature**

```python
fmriha.preprocessing.fileio.bids_filter(file_list, match_mode='exact', exclude_keys=None, exclude_values=None, **filters)
```

**Purpose**

Filter BIDS-style filenames based on flexible matching rules.

**Parameters**

| Parameter | Default | Meaning |
| --- | --- | --- |
| `file_list` | `required` | Input file list. |
| `match_mode` | `'exact'` | Matching mode. |
| `exclude_keys` | `None` | Exclude files that contain these keys (e.g., ['split', 'part']). |
| `exclude_values` | `None` | Exclude files whose *any* field contains these values. Example: `exclude_values=['seg1', 'seg2']`. |
| `**filters` | `optional` | Key-value pairs for filtering (e.g., sub='01', roi=['lh','rh']). |

**Returns or outputs**

matched_list : list of Path Files that match all rules.


## Preprocessing Workflows

`fmriha.preprocessing.prep`

Validation, conversion, and file-combination routines used before hyperalignment or statistical analysis.


### `input_validation_nii`

**Signature**

```python
fmriha.preprocessing.prep.input_validation_nii(file_list, ses='data', bids_parse=True, subject_key='sub', check_dim=True, group_by=None, role='train', njobs=3, verbose=True)
```

**Purpose**

Validate NIfTI or CIFTI input files and collect file metadata.

**Parameters**

| Parameter | Default | Meaning |
| --- | --- | --- |
| `file_list` | `required` | Input NIfTI/CIFTI filenames, for example `['sub-01_bold.nii.gz', 'sub-02_bold.dtseries.nii']`. |
| `ses` | `'data'` | Session label used by downstream output naming. |
| `bids_parse` | `True` | If True, parse BIDS-like entities from filenames. |
| `subject_key` | `'sub'` | Subject marker used when `bids_parse=False`. |
| `check_dim` | `True` | If True, compare data shapes across all files or within each group. |
| `group_by` | `None` | Filename/entity key used to group files before dimension checking, such as `'run'`. If None, all files are checked together. |
| `role` | `'train'` | Dataset role label used by downstream output naming. |
| `njobs` | `3` | Number of parallel workers. |
| `verbose` | `True` | If True, print validation progress. |

**Returns or outputs**

result : dict Validation summary, parsed entities, image shapes, missing files, and loading failures.

### `input_validation_gii`

**Signature**

```python
fmriha.preprocessing.prep.input_validation_gii(file_list, ses='data', bids_parse=True, subject_key='sub', check_dim=True, group_by='run', role='train', njobs=3, verbose=True)
```

**Purpose**

Validate GIFTI input files and collect per-hemisphere metadata.

**Parameters**

| Parameter | Default | Meaning |
| --- | --- | --- |
| `file_list` | `required` | Each item maps hemisphere labels to filenames. The same keys must be present in every item, for example `[{'l': 'sub-01_l.func.gii', 'r': 'sub-01_r.func.gii'}]`. |
| `ses` | `'data'` | Session label used by downstream output naming. |
| `bids_parse` | `True` | If True, parse BIDS-like entities from filenames. |
| `subject_key` | `'sub'` | Subject marker used when `bids_parse=False`. |
| `check_dim` | `True` | If True, compare GIFTI data shapes within each hemisphere and group. |
| `group_by` | `'run'` | Filename/entity key used to group files before dimension checking. |
| `role` | `'train'` | Dataset role label used by downstream output naming. |
| `njobs` | `3` | Number of parallel workers. |
| `verbose` | `True` | If True, print validation progress. |

**Returns or outputs**

result : dict Validation summary, per-hemisphere parsed entities, data shapes, missing files, and loading failures.

### `update_json`

**Signature**

```python
fmriha.preprocessing.prep.update_json(json_path, data, overwrite=False)
```

**Purpose**

Append one record to a JSON log while keeping a list-of-dict structure.

**Parameters**

| Parameter | Default | Meaning |
| --- | --- | --- |
| `json_path` | `required` | JSON file to create or update. |
| `data` | `required` | Record to write into the JSON log. |
| `overwrite` | `False` | If True, replace the existing log with `[data]`. If False, append `data` to the existing list. A single existing dict is converted to a two-item list. |

**Returns or outputs**

json_data : list Full JSON list after the update.

### `write_json_input_validate`

**Signature**

```python
fmriha.preprocessing.prep.write_json_input_validate(step_info, json_path, overwrite=False)
```

**Purpose**

Write input-validation results into a JSON log.

**Parameters**

| Parameter | Default | Meaning |
| --- | --- | --- |
| `step_info` | `required` | Result returned by `input_validation_nii` or `input_validation_gii`. |
| `json_path` | `required` | Output JSON filename. |
| `overwrite` | `False` | If True, clear previous records and write only this validation step. |

**Returns or outputs**

json_data : list Full JSON list after the update.

### `prep_surf`

**Signature**

```python
fmriha.preprocessing.prep.prep_surf(work_dir, input_info, do_resample=True, res_param=None, do_medial_rm=True, medial_param=None, normalize='zscore', dtype=np.float32, njobs=3, verbose=5, json_path='prep_log.json', overwrite=False, log_num='00001')
```

**Purpose**

Prepare surface fMRI data and save time-by-vertex matrices.

**Parameters**

| Parameter | Default | Meaning |
| --- | --- | --- |
| `work_dir` | `required` | Root directory for subject outputs and logs. |
| `input_info` | `required` | Validation result from `input_validation_nii` or `input_validation_gii`. Must contain `ses`, `role`, and `file_info`. |
| `do_resample` | `True` | If True, apply a surface mapper before masking. |
| `res_param` | `None` | Surface mapping parameters. `data` may be a provided mapper dict keyed by `'l'`/`'r'`. If `data` is None, mapper matrices are loaded from neuroboros with `source`, `target`, `source_mask`, and `target_mask`. `res` is used in output filenames. |
| `do_medial_rm` | `True` | If True, apply a target-space boolean mask, commonly a cortical mask that removes medial wall vertices. |
| `medial_param` | `None` | Mask parameters. `data` may be a provided mask dict keyed by `'l'`/`'r'`. If `data` is None, masks are loaded from neuroboros using `space`. `roi_name` is used in output filenames. |
| `normalize` | `'zscore'` | Optional normalization applied across time for each vertex. |
| `dtype` | `np.float32` | Data precision for loaded and saved arrays. |
| `njobs` | `3` | Number of parallel jobs. |
| `verbose` | `5` | Verbosity level passed to joblib. |
| `json_path` | `'prep_log.json'` | JSON file used to record processing parameters. |
| `overwrite` | `False` | Whether to overwrite the JSON log. |
| `log_num` | `'00001'` | Log file identifier. |

**Returns or outputs**

None Writes `.npy` matrices under `work_dir/sub-*/resample/response`.

### `prep_volume`

**Signature**

```python
fmriha.preprocessing.prep.prep_volume(work_dir, input_info, cifti_region='whole', roi_name='cortical_cift', do_resample=True, res_param=None, mask=None, save_resample_mask=True, normalize='zscore', dtype=np.float32, njobs=3, verbose=5, json_path='prep_log.json', overwrite=False, log_num='00001')
```

**Purpose**

Prepare volumetric fMRI data and save time-by-voxel matrices.

**Parameters**

| Parameter | Default | Meaning |
| --- | --- | --- |
| `work_dir` | `required` | Root working directory for outputs and logs. |
| `input_info` | `required` | Validation result from `input_validation_nii`. Must contain `ses`, `role`, and `file_info`. |
| `cifti_region` | `'whole'` | CIFTI subcortical structure selector passed to `fio.get_cifti_subcortical`. |
| `roi_name` | `'cortical_cift'` | ROI label used in output folders and filenames. |
| `do_resample` | `True` | If True, resample input volumes to a reference image or voxel size. |
| `res_param` | `None` | Parameters for `fio.nifti_resample`. Expected keys are `ref`, `interpolation`, `vol_size`, and `mode` when `do_resample=True`. `res` is used in output filenames. |
| `mask` | `None` | External 3D mask for NIfTI input. If needed, it is resliced to the final image grid used for extraction. |
| `save_resample_mask` | `True` | If True, save a representative 3D mask used for extraction. |
| `normalize` | `'zscore'` | Optional normalization method passed to `fio.stand_matrix`. |
| `dtype` | `np.float32` | Data precision for numeric arrays. |
| `njobs` | `3` | Number of parallel jobs for file-wise processing. |
| `verbose` | `5` | Verbosity level for joblib. |
| `json_path` | `'prep_log.json'` | Path to a JSON file where processing records are stored. |
| `overwrite` | `False` | Whether to overwrite existing JSON records with the same step. |
| `log_num` | `'00001'` | Log identifier suffix. |

**Returns or outputs**

None Writes .npy time-by-voxel matrices and a 3D mask NIfTI to disk.

### `combine_file`

**Signature**

```python
fmriha.preprocessing.prep.combine_file(work_dir, sub_list, step, modality, structures, ses_name, njobs, method='zscore', json_path='prep_log.json', dtype='np.float32', overwrite=False, log_num='00001', rename_fields=None, **file_filter)
```

**Purpose**

Concatenate multiple files for each subject and structure.

**Parameters**

| Parameter | Default | Meaning |
| --- | --- | --- |
| `work_dir` | `required` | Root working directory containing subject folders. |
| `sub_list` | `required` | Subject folder names, such as `['sub-01', 'sub-02']`. |
| `step` | `required` | Processing step folder, such as `'resample'`. |
| `modality` | `required` | Data modality folder, such as `'response'` or `'connectivity'`. `'mixed'` is not supported by this function. |
| `structures` | `required` | Structure folders to process, such as `'hemi-L'` or `['hemi-L', 'hemi-R']`. |
| `ses_name` | `required` | Session label used to rename the combined output. |
| `njobs` | `required` | Number of parallel jobs. |
| `method` | `'zscore'` | Optional normalization applied after concatenation. |
| `json_path` | `'prep_log.json'` | Path to JSON log file. |
| `dtype` | `'np.float32'` | Output precision used when loading and normalizing arrays. The legacy string `'np.float32'` is accepted. |
| `overwrite` | `False` | Whether to overwrite existing log entries. |
| `log_num` | `'00001'` | Log file identifier. |
| `rename_fields` | `None` | BIDS fields to replace in the output filename. If None, `{'ses': ses_name}` is used. |
| `**file_filter` | `optional` | Filters passed to `fio.bids_filter`. |

**Returns or outputs**

None Saves concatenated files and updates logs and JSON record.


## Searchlight Downsampling

`fmriha.gensls.dsample`

Sampling utilities for selecting surface or volume searchlight centers.


### `fibonacci_sphere`

**Signature**

```python
fmriha.gensls.dsample.fibonacci_sphere(samples=500)
```

**Purpose**

Generate approximately uniform points on a unit sphere.

**Parameters**

| Parameter | Default | Meaning |
| --- | --- | --- |
| `samples` | `500` | Number of points to generate. Must be at least 2. |

**Returns or outputs**

ndarray of shape (samples, 3) Sphere coordinates.

### `find_closest_vertices`

**Signature**

```python
fmriha.gensls.dsample.find_closest_vertices(fibonacci_points, vertices)
```

**Purpose**

Match each Fibonacci sphere point to one unique nearest vertex.

**Parameters**

| Parameter | Default | Meaning |
| --- | --- | --- |
| `fibonacci_points` | `required` | Sample points on the sphere. |
| `vertices` | `required` | Vertex coordinates. |

**Returns or outputs**

list of int Selected vertex indices in the same order as `fibonacci_points`.

### `load_cifti_subcortical`

**Signature**

```python
fmriha.gensls.dsample.load_cifti_subcortical(img)
```

**Purpose**

Collect voxel-based brain-model columns and IJK coordinates from a CIFTI.

**Parameters**

| Parameter | Default | Meaning |
| --- | --- | --- |
| `img` | `required` | Loaded CIFTI image. |

**Returns or outputs**

subcol_indices : ndarray of shape (M,) Global column indices in the CIFTI dense array corresponding to voxel data. voxel_ijk : ndarray of shape (M, 3) Integer voxel indices in the CIFTI volume grid.

### `fps_trim`

**Signature**

```python
fmriha.gensls.dsample.fps_trim(X, keep_indices, K, seed=616)
```

**Purpose**

Run farthest-point sampling within a candidate subset.

**Parameters**

| Parameter | Default | Meaning |
| --- | --- | --- |
| `X` | `required` | Point cloud. |
| `keep_indices` | `required` | Candidate indices into `X`. |
| `K` | `required` | Number of points to keep. |
| `seed` | `616` | Random seed for the initial point. |

**Returns or outputs**

ndarray of shape (K,) Indices relative to `keep_indices`.

### `fps_grow_with_seeds`

**Signature**

```python
fmriha.gensls.dsample.fps_grow_with_seeds(X, seed_indices, K_total, seed=616)
```

**Purpose**

Expand an initial point set with farthest-point sampling.

**Parameters**

| Parameter | Default | Meaning |
| --- | --- | --- |
| `X` | `required` | Point cloud. |
| `seed_indices` | `required` | Already-selected global indices into `X`. May be empty. |
| `K_total` | `required` | Desired total number of selected points. |
| `seed` | `616` | Random seed used only when no initial seeds are supplied. |

**Returns or outputs**

ndarray of shape (K_total,) Global indices into `X`.

### `downsample_cifti_subcortical`

**Signature**

```python
fmriha.gensls.dsample.downsample_cifti_subcortical(cifti_file, N=None, mask3d_out=None, sls_type='subcortex', voxel_sizes=None, drop_all_zero_columns=True, whole_mask=False, hemi_mask=True)
```

**Purpose**

Downsample subcortical voxel centers from a CIFTI file.

**Parameters**

| Parameter | Default | Meaning |
| --- | --- | --- |
| `cifti_file` | `required` | Input dense CIFTI file. |
| `N` | `None` | Desired number of centers in downsample mode. Ignored when `sls_type="seed"`. |
| `mask3d_out` | `None` | Output folder. If provided, masks are saved to disk and the function returns `None`. |
| `sls_type` | `'subcortex'` | Searchlight-center mode. `"seed"` keeps all voxel centers. Any other value uses downsampling. |
| `voxel_sizes` | `None` | Voxel spacing used when computing geometric distances. |
| `drop_all_zero_columns` | `True` | Whether to discard voxel columns that are all zero across samples. |
| `whole_mask` | `False` | Whether to save the whole subcortical mask when `mask3d_out` is set. |
| `hemi_mask` | `True` | Whether to save split left/right/brainstem masks when `mask3d_out` is set. |

**Returns or outputs**

dict or None If `mask3d_out` is None, returns a dictionary with keys `whole`, `l`, `r`, `brain_stem`, and `meta`. Otherwise returns `None` after saving output masks.

### `downsample_nifti_volume`

**Signature**

```python
fmriha.gensls.dsample.downsample_nifti_volume(mask_data, N=None, voxel_sizes=None, bins_per_axis=16, alpha_count=0.005, modulus=8, beam_width=12, max_offsets=64, rng_seed=616)
```

**Purpose**

Performs the operation indicated by the function name and arguments.

**Parameters**

| Parameter | Default | Meaning |
| --- | --- | --- |
| `mask_data` | `required` | 3D mask array used as the candidate volume for downsampling. |
| `N` | `None` | Target number of samples, centers, or retained elements. |
| `voxel_sizes` | `None` | Voxel size vector used to convert voxel indices into physical distances. |
| `bins_per_axis` | `16` | Number of spatial bins per axis used during volume downsampling. |
| `alpha_count` | `0.005` | Weight assigned to bin-count balance during volume downsampling. |
| `modulus` | `8` | Grid modulus used to build candidate offsets for volume center selection. |
| `beam_width` | `12` | Number of candidate solutions retained during beam-style search. |
| `max_offsets` | `64` | Maximum number of grid offsets considered during downsampling. |
| `rng_seed` | `616` | Random seed used for reproducible sampling. |

**Returns or outputs**

See the function output or saved pipeline records for returned objects and generated files.


## Searchlight Generation

`fmriha.gensls.gensearchlight`

Surface and volume searchlight construction utilities.


### `compute_neighbors`

**Signature**

```python
fmriha.gensls.gensearchlight.compute_neighbors(faces, nv=None)
```

**Purpose**

Build the 1-hop neighbor list for each mesh vertex.

**Parameters**

| Parameter | Default | Meaning |
| --- | --- | --- |
| `faces` | `required` | Triangle definitions. |
| `nv` | `None` | Total number of vertices. If None, it is inferred from `faces`. |

**Returns or outputs**

list of ndarray Neighbor vertex indices for each vertex.

### `compute_neighbor_distances`

**Signature**

```python
fmriha.gensls.gensearchlight.compute_neighbor_distances(coords, neighbors)
```

**Purpose**

Compute Euclidean distances from each vertex to its neighbors.

**Parameters**

| Parameter | Default | Meaning |
| --- | --- | --- |
| `coords` | `required` | Vertex coordinates. |
| `neighbors` | `required` | Neighbor lists for each vertex. |

**Returns or outputs**

list of ndarray Neighbor distances aligned with `neighbors`.

### `dijkstra_distances`

**Signature**

```python
fmriha.gensls.gensearchlight.dijkstra_distances(src, nv, neighbors, neighbor_distances, max_dist=None)
```

**Purpose**

Compute Dijkstra distances from one source vertex.

**Parameters**

| Parameter | Default | Meaning |
| --- | --- | --- |
| `src` | `required` | Source vertex index. |
| `nv` | `required` | Total number of vertices. |
| `neighbors` | `required` | Neighbor list for each vertex. |
| `neighbor_distances` | `required` | Edge lengths aligned with `neighbors`. |
| `max_dist` | `None` | Optional search cutoff. |

**Returns or outputs**

ndarray of shape (nv,) Dijkstra distances from `src`.

### `dijkstra`

**Signature**

```python
fmriha.gensls.gensearchlight.dijkstra(src, nv, neighbors, neighbor_distances, max_dist, sort=True)
```

**Purpose**

Compute searchlight members and distances within a Dijkstra radius.

**Parameters**

| Parameter | Default | Meaning |
| --- | --- | --- |
| `src` | `required` | Source vertex index. |
| `nv` | `required` | Total number of vertices. |
| `neighbors` | `required` | Neighbor list for each vertex. |
| `neighbor_distances` | `required` | Edge lengths aligned with `neighbors`. |
| `max_dist` | `required` | Maximum geodesic distance. |
| `sort` | `True` | Whether to sort members by distance. |

**Returns or outputs**

dijkstra_nbrs : ndarray Vertex indices within `max_dist`. dijkstra_dists : ndarray Distances for `dijkstra_nbrs`.

### `get_surfgii_data`

**Signature**

```python
fmriha.gensls.gensearchlight.get_surfgii_data(surfgii_file)
```

**Purpose**

Load coordinates and faces from one or more surface GIFTI files.

**Parameters**

| Parameter | Default | Meaning |
| --- | --- | --- |
| `surfgii_file` | `required` | Mapping from hemisphere key to GIFTI filepath. |

**Returns or outputs**

dict Nested dictionary with `coord` and `faces` arrays per key.

### `generate_searchlight_surf`

**Signature**

```python
fmriha.gensls.gensearchlight.generate_searchlight_surf(lr, surface_file, center_file, radius, cortical_mask)
```

**Purpose**

Generate surface searchlights from dense or downsampled centers.

**Parameters**

| Parameter | Default | Meaning |
| --- | --- | --- |
| `lr` | `required` | Hemisphere key. |
| `surface_file` | `required` | Dense surface mesh data. |
| `center_file` | `required` | Optional downsampled center surface data. |
| `radius` | `required` | Searchlight radius. |
| `cortical_mask` | `required` | Optional Boolean vertex mask. |

**Returns or outputs**

tuple `(searchlights, distances)`.

### `sls_update`

**Signature**

```python
fmriha.gensls.gensearchlight.sls_update(sls, dists=None, medial_wall=None, lr=None, center_type='first')
```

**Purpose**

Remove medial-wall vertices and update indices for existing searchlights.

**Parameters**

| Parameter | Default | Meaning |
| --- | --- | --- |
| `sls` | `required` | Searchlight members for each hemisphere. |
| `dists` | `None` | Distances corresponding to `sls`. |
| `medial_wall` | `None` | Boolean mask per hemisphere, where True indicates medial-wall vertices. |
| `lr` | `None` | Hemisphere keys to process. |
| `center_type` | `'first'` | Strategy used to determine each searchlight center. |

**Returns or outputs**

tuple `(sls_new, dists_new)` where `dists_new` is None if `dists` is None.

### `annular_sls`

**Signature**

```python
fmriha.gensls.gensearchlight.annular_sls(radius_center, radius_halfwidth, func_type, lr, surface_file, cortical_mask=None)
```

**Purpose**

Build annular surface searchlights by subtracting two nested radii.

**Parameters**

| Parameter | Default | Meaning |
| --- | --- | --- |
| `radius_center` | `required` | Annulus center radius. |
| `radius_halfwidth` | `required` | Half-width of the annulus. |
| `func_type` | `required` | Searchlight backend selector. Supported values are the existing project conventions: `lr` and `"surf_dense"`. |
| `lr` | `required` | Hemisphere keys. |
| `surface_file` | `required` | Surface specification matching `func_type`. |
| `cortical_mask` | `None` | Optional Boolean cortical mask. |

**Returns or outputs**

dict Annular searchlights by hemisphere.

### `volume_searchlight_dist`

**Signature**

```python
fmriha.gensls.gensearchlight.volume_searchlight_dist(mask, center_coordinate, radius)
```

**Purpose**

Generate one volumetric searchlight around a center voxel.

**Parameters**

| Parameter | Default | Meaning |
| --- | --- | --- |
| `mask` | `required` | Binary or integer mask defining valid voxels. |
| `center_coordinate` | `required` | Center voxel coordinate. |
| `radius` | `required` | Euclidean searchlight radius in voxel units. |

**Returns or outputs**

coord_out_f : ndarray Searchlight voxel coordinates sorted by distance. distance_out_f : ndarray Distances for `coord_out_f`. index_out : list of int Linear-in-mask indices corresponding to `coord_out_f`.

### `generate_searchlight_vol`

**Signature**

```python
fmriha.gensls.gensearchlight.generate_searchlight_vol(mask_dense, mask_center=None, radius=2, threshold=0, njobs=0, verbose=0)
```

**Purpose**

Generate volumetric searchlights from dense or downsampled center masks.

**Parameters**

| Parameter | Default | Meaning |
| --- | --- | --- |
| `mask_dense` | `required` | 3D mask defining valid voxels for searchlights. |
| `mask_center` | `None` | Optional 3D mask defining center voxels. If None, dense centers are used. |
| `radius` | `2` | Searchlight radius in voxel units. |
| `threshold` | `0` | Minimum in-mask fraction required to retain a searchlight. |
| `njobs` | `0` | Number of parallel workers. `0` runs sequentially. |
| `verbose` | `0` | Joblib verbosity level. |

**Returns or outputs**

dict Dictionary with keys `sls_coord`, `sls_idx`, and `dists`.


## Linear Algebra

`fmriha.src.linalg`

Low-level SVD and PCA helpers.


### `safe_svd`

**Signature**

```python
fmriha.src.linalg.safe_svd(X, remove_mean=True)
```

**Purpose**

Singular value decomposition without occasional LinAlgError crashes.

**Parameters**

| Parameter | Default | Meaning |
| --- | --- | --- |
| `X` | `required` | The matrix to be decomposed in NumPy array format. |
| `remove_mean` | `True` | Whether to subtract the mean of each column before the actual SVD (True) or not (False). Setting `remove_mean=True` is helpful when the SVD is used to perform PCA. |

**Returns or outputs**

U : ndarray of shape (M, K) Unitary matrix. s : ndarray of shape (K,) The singular values. Vt : ndarray of shape (K, N) Unitary matrix.

### `svd_pca`

**Signature**

```python
fmriha.src.linalg.svd_pca(X, remove_mean=True)
```

**Purpose**

Principal component analysis (PCA) based on SVD.

**Parameters**

| Parameter | Default | Meaning |
| --- | --- | --- |
| `X` | `required` | The data matrix to be transformed into PC space. |
| `remove_mean` | `True` | Whether to subtract the mean of each column before the SVD (True) or not (False). This parameter should be set to True unless the columns already have zero mean. |

**Returns or outputs**

X_new : ndarray of shape (M, K) The transformed data matrix in PC space.


## Procrustes Alignment

`fmriha.src.procrustes`

Single-region Procrustes transformation estimation.


### `procrustes`

**Signature**

```python
fmriha.src.procrustes.procrustes(X, Y, reflection=True, scaling=False)
```

**Purpose**

The orthogonal Procrustes algorithm.

**Parameters**

| Parameter | Default | Meaning |
| --- | --- | --- |
| `X` | `required` | The data matrix to be aligned to Y. |
| `Y` | `required` | The "target" data matrix -- the matrix to be aligned to. |
| `reflection` | `True` | Whether allows reflection in the transformation (True) or not (False). Note that even with `reflection=True`, the solution may not contain a reflection if the alignment cannot be improved by adding a reflection to the rotation. |
| `scaling` | `False` | Whether to allow a global scale factor in the Procrustes transformation. This can improve alignment quality while changing overall response scale. |

**Returns or outputs**

T : ndarray The transformation matrix which can be used to align X to Y. Depending on the parameters `reflection` and `scaling`, the transformation can be a pure rotation, an improper rotation, or a pure/improper rotation with global scaling.


## Local Templates

`fmriha.src.local_template`

Template construction routines used inside hyperalignment.

These functions operate on subject-by-time-by-feature arrays and are normally called by higher-level hyperalignment workflows.


### `PCA_decomposition`

**Signature**

```python
fmriha.src.local_template.PCA_decomposition(dss, max_npc=None, flavor='sklearn', adjust_ns=False, demean=True, only_evr=False)
```

**Purpose**

se2eDecompose concatenated data matrices using PCA/SVD.

**Parameters**

| Parameter | Default | Meaning |
| --- | --- | --- |
| `dss` | `required` | Subject data stack, usually shaped as subjects by time points by features. |
| `max_npc` | `None` | Maximum number of principal components retained in PCA-based templates. |
| `flavor` | `'sklearn'` | Implementation backend for PCA or SVD computation. |
| `adjust_ns` | `False` | Whether to adjust the variance of the output so that it doesn't increase with the number of subjects. |
| `demean` | `True` | Whether to remove the mean of the columns prior to SVD. |
| `only_evr` | `False` | Whether to return explained-variance ratios instead of a template matrix. |

**Returns or outputs**

XX : ndarray of shape (nt, npc) cc : ndarray of shape (npc, ns, nv)

### `compute_PCA_template`

**Signature**

```python
fmriha.src.local_template.compute_PCA_template(dss, sl=None, max_npc=None, flavor='sklearn', demean=True, only_evr=False)
```

**Purpose**

Performs the operation indicated by the function name and arguments.

**Parameters**

| Parameter | Default | Meaning |
| --- | --- | --- |
| `dss` | `required` | Subject data stack, usually shaped as subjects by time points by features. |
| `sl` | `None` | Optional searchlight index array used to select a local feature subset. |
| `max_npc` | `None` | Maximum number of principal components retained in PCA-based templates. |
| `flavor` | `'sklearn'` | Implementation backend for PCA or SVD computation. |
| `demean` | `True` | Whether to remove the mean before decomposition or alignment. |
| `only_evr` | `False` | Whether to return explained-variance ratios instead of a template matrix. |

**Returns or outputs**

See the function output or saved pipeline records for returned objects and generated files.

### `compute_PCA_var1_template`

**Signature**

```python
fmriha.src.local_template.compute_PCA_var1_template(dss, sl=None, max_npc=None, flavor='sklearn', demean=True)
```

**Purpose**

Performs the operation indicated by the function name and arguments.

**Parameters**

| Parameter | Default | Meaning |
| --- | --- | --- |
| `dss` | `required` | Subject data stack, usually shaped as subjects by time points by features. |
| `sl` | `None` | Optional searchlight index array used to select a local feature subset. |
| `max_npc` | `None` | Maximum number of principal components retained in PCA-based templates. |
| `flavor` | `'sklearn'` | Implementation backend for PCA or SVD computation. |
| `demean` | `True` | Whether to remove the mean before decomposition or alignment. |

**Returns or outputs**

See the function output or saved pipeline records for returned objects and generated files.

### `compute_PCA_var2_template`

**Signature**

```python
fmriha.src.local_template.compute_PCA_var2_template(dss, sl=None, max_npc=None, flavor='sklearn', demean=True)
```

**Purpose**

Performs the operation indicated by the function name and arguments.

**Parameters**

| Parameter | Default | Meaning |
| --- | --- | --- |
| `dss` | `required` | Subject data stack, usually shaped as subjects by time points by features. |
| `sl` | `None` | Optional searchlight index array used to select a local feature subset. |
| `max_npc` | `None` | Maximum number of principal components retained in PCA-based templates. |
| `flavor` | `'sklearn'` | Implementation backend for PCA or SVD computation. |
| `demean` | `True` | Whether to remove the mean before decomposition or alignment. |

**Returns or outputs**

See the function output or saved pipeline records for returned objects and generated files.

### `compute_procrustes_template`

**Signature**

```python
fmriha.src.local_template.compute_procrustes_template(dss, sl=None, reflection=True, scaling=False, zscore_common=True, level2_iter=1, dss2=None, start_point=0, **kwargs)
```

**Purpose**

Performs the operation indicated by the function name and arguments.

**Parameters**

| Parameter | Default | Meaning |
| --- | --- | --- |
| `dss` | `required` | Subject data stack, usually shaped as subjects by time points by features. |
| `sl` | `None` | Optional searchlight index array used to select a local feature subset. |
| `reflection` | `True` | Whether to allow reflection in the Procrustes transformation. |
| `scaling` | `False` | Whether to allow a global scale factor in the transformation. |
| `zscore_common` | `True` | Whether to z-score the common template after Procrustes template estimation. |
| `level2_iter` | `1` | Number of second-level template-refinement iterations. |
| `dss2` | `None` | Optional second data stack used for template initialization or refinement. |
| `start_point` | `0` | Subject index used as the initial reference point. |
| `**kwargs` | `optional` | Additional keyword arguments passed to the underlying routine. |

**Returns or outputs**

See the function output or saved pipeline records for returned objects and generated files.

### `compute_template`

**Signature**

```python
fmriha.src.local_template.compute_template(dss, sl=None, kind='procrustes', max_npc=None, common_topography=False, demean=False, start_point=0, only_evr=False)
```

**Purpose**

Performs the operation indicated by the function name and arguments.

**Parameters**

| Parameter | Default | Meaning |
| --- | --- | --- |
| `dss` | `required` | Subject data stack, usually shaped as subjects by time points by features. |
| `sl` | `None` | Optional searchlight index array used to select a local feature subset. |
| `kind` | `'procrustes'` | Configuration or algorithm kind requested by the function. |
| `max_npc` | `None` | Maximum number of principal components retained in PCA-based templates. |
| `common_topography` | `False` | Whether to preserve a shared topographic basis during template construction. |
| `demean` | `False` | Whether to remove the mean before decomposition or alignment. |
| `start_point` | `0` | Subject index used as the initial reference point. |
| `only_evr` | `False` | Whether to return explained-variance ratios instead of a template matrix. |

**Returns or outputs**

See the function output or saved pipeline records for returned objects and generated files.


## Searchlight Alignment Primitives

`fmriha.src.searchlight`

Local searchlight transforms and searchlight template estimation.

These are lower-level routines for aligning one pair of matrices across many local neighborhoods.


### `ridge`

**Signature**

```python
fmriha.src.searchlight.ridge(X, Y, alpha=1000.0)
```

**Purpose**

Return a ridge-regression transform mapping `X` to `Y`.

**Parameters**

| Parameter | Default | Meaning |
| --- | --- | --- |
| `X` | `required` | Input data matrix or point cloud. |
| `Y` | `required` | Target data matrix. |
| `alpha` | `1000.0` | Significance or regularization parameter, depending on the function. |

**Returns or outputs**

See the function output or saved pipeline records for returned objects and generated files.

### `compute_searchlight_weights`

**Signature**

```python
fmriha.src.searchlight.compute_searchlight_weights(sls, dists, radius)
```

**Purpose**

weights = compute_searchlight_weights(sls, dists, radius)

**Parameters**

| Parameter | Default | Meaning |
| --- | --- | --- |
| `sls` | `required` | Searchlight definitions; each item contains the feature indices in one local neighborhood. |
| `dists` | `required` | Distances associated with searchlight members. |
| `radius` | `required` | Searchlight radius. |

**Returns or outputs**

See the function output or saved pipeline records for returned objects and generated files.

### `searchlight_hyperalignment`

**Signature**

```python
fmriha.src.searchlight.searchlight_hyperalignment(X, Y, sls, dists, radius, T0, sl_func, weighted=True)
```

**Purpose**

Performs the operation indicated by the function name and arguments.

**Parameters**

| Parameter | Default | Meaning |
| --- | --- | --- |
| `X` | `required` | Input data matrix or point cloud. |
| `Y` | `required` | Target data matrix. |
| `sls` | `required` | Searchlight definitions; each item contains the feature indices in one local neighborhood. |
| `dists` | `required` | Distances associated with searchlight members. |
| `radius` | `required` | Searchlight radius. |
| `T0` | `required` | Initial or accumulated transformation matrix used by searchlight alignment. |
| `sl_func` | `required` | Local alignment function applied to each searchlight. |
| `weighted` | `True` | Whether to weight local searchlight contributions by distance-based weights. |

**Returns or outputs**

See the function output or saved pipeline records for returned objects and generated files.

### `searchlight_procrustes`

**Signature**

```python
fmriha.src.searchlight.searchlight_procrustes(X, Y, sls, dists, radius, T0=None, reflection=True, scaling=False, weighted=True)
```

**Purpose**

Performs the operation indicated by the function name and arguments.

**Parameters**

| Parameter | Default | Meaning |
| --- | --- | --- |
| `X` | `required` | Input data matrix or point cloud. |
| `Y` | `required` | Target data matrix. |
| `sls` | `required` | Searchlight definitions; each item contains the feature indices in one local neighborhood. |
| `dists` | `required` | Distances associated with searchlight members. |
| `radius` | `required` | Searchlight radius. |
| `T0` | `None` | Initial or accumulated transformation matrix used by searchlight alignment. |
| `reflection` | `True` | Whether to allow reflection in the Procrustes transformation. |
| `scaling` | `False` | Whether to allow a global scale factor in the transformation. |
| `weighted` | `True` | Whether to weight local searchlight contributions by distance-based weights. |

**Returns or outputs**

See the function output or saved pipeline records for returned objects and generated files.

### `searchlight_ridge`

**Signature**

```python
fmriha.src.searchlight.searchlight_ridge(X, Y, sls, dists, radius, T0=None, alpha=1000.0, weighted=True)
```

**Purpose**

Performs the operation indicated by the function name and arguments.

**Parameters**

| Parameter | Default | Meaning |
| --- | --- | --- |
| `X` | `required` | Input data matrix or point cloud. |
| `Y` | `required` | Target data matrix. |
| `sls` | `required` | Searchlight definitions; each item contains the feature indices in one local neighborhood. |
| `dists` | `required` | Distances associated with searchlight members. |
| `radius` | `required` | Searchlight radius. |
| `T0` | `None` | Initial or accumulated transformation matrix used by searchlight alignment. |
| `alpha` | `1000.0` | Significance or regularization parameter, depending on the function. |
| `weighted` | `True` | Whether to weight local searchlight contributions by distance-based weights. |

**Returns or outputs**

See the function output or saved pipeline records for returned objects and generated files.

### `searchlight_template`

**Signature**

```python
fmriha.src.searchlight.searchlight_template(dss, sls, dists, radius, n_jobs=1, tmpl_kind='pca')
```

**Purpose**

Performs the operation indicated by the function name and arguments.

**Parameters**

| Parameter | Default | Meaning |
| --- | --- | --- |
| `dss` | `required` | Subject data stack, usually shaped as subjects by time points by features. |
| `sls` | `required` | Searchlight definitions; each item contains the feature indices in one local neighborhood. |
| `dists` | `required` | Distances associated with searchlight members. |
| `radius` | `required` | Searchlight radius. |
| `n_jobs` | `1` | Number of parallel jobs. |
| `tmpl_kind` | `'pca'` | Template-construction method used for searchlight templates. |

**Returns or outputs**

See the function output or saved pipeline records for returned objects and generated files.


## Sparse Matrix Utilities

`fmriha.src.sparse`

Sparse matrix initialization for searchlight aggregation.


### `initialize_sparse_matrix`

**Signature**

```python
fmriha.src.sparse.initialize_sparse_matrix(sls, nv=None, dtype=np.float64, cache_fn=None)
```

**Purpose**

Initialize a sparse matrix based on the searchlights.

**Parameters**

| Parameter | Default | Meaning |
| --- | --- | --- |
| `sls` | `required` | A list of searchlights. Each entry is an integer array comprising the indices of a searchlight. |
| `nv` | `None` | Number of vertices, by default None. The sparse matrix has a shape of (nv, nv). |
| `dtype` | `np.float64` | The dtype of the generated sparse matrix, by default np.float64 |
| `cache_fn` | `None` | The file name of the cached sparse matrix. If the file exists, it will be loaded to avoid repeated computations. If it is `None`, no cached file will be used and the matrix will be computed. |

**Returns or outputs**

mat : csc_matrix The initialized sparse matrix which allows fast computations for searchlight algorithms. All of its elements are 0's.


## Core Hyperalignment

`fmriha.ha.core`

Core common-space, transformation, and searchlight aggregation functions.


### `cspace_sl`

**Signature**

```python
fmriha.ha.core.cspace_sl(sub_file_list, sl, cspace_kind, common_topography, demean=True, start_sub=0)
```

**Purpose**

Compute the common space for one searchlight.

**Parameters**

| Parameter | Default | Meaning |
| --- | --- | --- |
| `sub_file_list` | `required` | Subject-level `.npy` files. Each file must contain a 2D matrix with shape `(n_time, n_vertex)`. |
| `sl` | `required` | Vertex or voxel indices belonging to one searchlight. |
| `cspace_kind` | `required` | Common-template method. Supported values are `'procrustes'`, `'pca'`, `'pcav1'`, `'pcav2'`, and `'cls'`. |
| `common_topography` | `required` | Whether to project the common space back to vertex/voxel topography. |
| `demean` | `True` | Whether to remove the mean of the columns prior to SVD. |
| `start_sub` | `0` | Reference subject for `cspace_kind='procrustes'`. Use `'mean'` to initialize the reference from the mean of all subjects. |

**Returns or outputs**

cspace_single : ndarray Common space for this searchlight. sl : ndarray Searchlight indices, returned unchanged except for integer dtype.

### `xform_sl`

**Signature**

```python
fmriha.ha.core.xform_sl(sub_data, cspace, sl, reflection=True, scaling=False)
```

**Purpose**

Compute the transformation matrix for a single searchlight.

**Parameters**

| Parameter | Default | Meaning |
| --- | --- | --- |
| `sub_data` | `required` | Raw subject data to be aligned, with shape `(n_time, n_vertex)`. |
| `cspace` | `required` | Common template to align to, with the same shape as `sub_data`. |
| `sl` | `required` | Vertex or voxel indices belonging to one searchlight. |
| `reflection` | `True` | Whether to allow reflections in the transformation. |
| `scaling` | `False` | Whether to allow scaling in the transformation. |

**Returns or outputs**

m : ndarray Transformation matrix for the single searchlight. sl : ndarray Searchlight indices, returned unchanged except for integer dtype.

### `aggregate_sls_m`

**Signature**

```python
fmriha.ha.core.aggregate_sls_m(sls_result, sls, weights, nv, dtype=np.float32, chunk_size=2000)
```

**Purpose**

Aggregate searchlight transformation matrices into a global matrix.

**Parameters**

| Parameter | Default | Meaning |
| --- | --- | --- |
| `sls_result` | `required` | Searchlight-level transformation matrices. Each matrix must have shape `(n_sl_vertex, n_sl_vertex)`. |
| `sls` | `required` | Searchlight index arrays. |
| `weights` | `required` | Vertex weights for each searchlight. Each weight vector must have the same length as the corresponding searchlight. |
| `nv` | `required` | Number of vertices or voxels in the full structure. |
| `dtype` | `np.float32` | Data precision used for accumulation. |
| `chunk_size` | `2000` | Number of searchlights accumulated before flushing to sparse format in chunked mode. |

**Returns or outputs**

T_sparse : csr_matrix Sparse global transformation matrix with shape `(nv, nv)`.

### `precompute_batch_template`

**Signature**

```python
fmriha.ha.core.precompute_batch_template(sls_batch, nv)
```

**Purpose**

Precompute sparse matrix coordinates for a batch of searchlights.

**Parameters**

| Parameter | Default | Meaning |
| --- | --- | --- |
| `sls_batch` | `required` | Searchlight index arrays. |
| `nv` | `required` | Number of vertices or voxels in the full structure. |

**Returns or outputs**

template : dict Coordinate and grouping arrays used by downstream sparse aggregation.


## Common-Space Construction

`fmriha.ha.cspace`

Pipeline-level common-space estimation over searchlights.


### `cspace_sls`

**Signature**

```python
fmriha.ha.cspace.cspace_sls(work_dir, sls, dists, radius, sub_list, task_name='rha', cspace_kind='procrustes', common_topography=False, weight=True, demean=True, start_sub=0, chunk_size=2000, njobs=5, step='resample', modality='response', structure_name='hemi-L', dtype=np.float32, scope='sls', save_local=False, suffix=None, json_path='prep_log.json', overwrite=False, log_num='00001', dask=False, cluster=None, **file_filters)
```

**Purpose**

Compute searchlight-level common spaces across subjects.

**Parameters**

| Parameter | Default | Meaning |
| --- | --- | --- |
| `work_dir` | `required` | Root working directory containing subject data and output folders. |
| `sls` | `required` | Searchlight definitions. Each element is a list/array of vertex or voxel indices belonging to one searchlight. If a dict is provided, keys should correspond to structure names (e.g., 'hemi-L', 'hemi-R', 'l', 'r'). |
| `dists` | `required` | Distances between the searchlight center and its vertices/voxels. Required when `weight=True`. Keys must match those of `sls` if a dict is provided. |
| `radius` | `required` | Searchlight radius. For surface data, interpreted as geodesic distance (in mm); for volumetric data, interpreted as voxel steps. |
| `sub_list` | `required` | List of subject identifiers (e.g., ['sub-01', 'sub-02']). |
| `task_name` | `'rha'` | Name used to label the output common space (default: 'rha'). |
| `cspace_kind` | `'procrustes'` | Method used to compute the common space. - 'pca': principal component analysis. - 'procrustes': iterative Procrustes alignment. |
| `common_topography` | `False` | Whether to project PCA components back to vertex/voxel space. Must be True when `cspace_kind='pca'`. |
| `weight` | `True` | Whether to weight vertices/voxels within each searchlight according to their distance from the center. |
| `demean` | `True` | Whether to remove column-wise means before SVD or alignment. |
| `start_sub` | `0` | Reference subject index for Procrustes alignment, or 'mean' to use the group mean as initialization. Only used when `cspace_kind='procrustes'`. |
| `chunk_size` | `2000` | Number of searchlights processed in one batch. Reduce this value if memory is limited. |
| `njobs` | `5` | Number of parallel workers. |
| `step` | `'resample'` | Name of the preprocessing step folder under each subject directory. |
| `modality` | `'response'` | Name of the modality folder under `step`. |
| `structure_name` | `'hemi-L'` | Structure(s) for which common spaces are computed (e.g., 'hemi-L'). |
| `dtype` | `np.float32` | Data type used for the output common space. |
| `scope` | `'sls'` | Label used to define the output filename (default: 'sls'). |
| `save_local` | `False` | Whether to save local searchlight-level common spaces or chunk-level intermediate results. |
| `suffix` | `None` | Additional suffix appended to the output filename. |
| `json_path` | `'prep_log.json'` | Path to the JSON file where processing parameters are recorded. |
| `overwrite` | `False` | Whether to overwrite existing entries in the JSON log. |
| `log_num` | `'00001'` | Identifier for the log file (e.g., '00001' → 'progress_00001.log'). |
| `dask` | `False` | Whether to use Dask-based distributed computation instead of joblib. |
| `cluster` | `None` | Dask cluster or cluster-like object used when `dask=True`. |
| `**file_filters` | `optional` | Keyword arguments used to filter input files following BIDS-style naming (e.g., run='01'). |

**Returns or outputs**

None The computed common spaces are saved as `.npy` files in the output directory defined under `work_dir`.


## Transformation Estimation

`fmriha.ha.xforms`

Subject-specific searchlight transformation workflows.


### `xform_sls_con`

**Signature**

```python
fmriha.ha.xforms.xform_sls_con(work_dir, sls, dists, radius, sub_list, source_step='resample', modality='response', structure_name=['hemi-L', 'hemi-R'], task_name='rha', cspace_desc=None, reflection=True, scaling=False, weight=True, chunk_size=2000, dtype=np.float32, njobs=5, scope='sls', suffix=None, json_path='prep_log.json', overwrite=False, log_num='00001', **file_filters)
```

**Purpose**

Compute subject-specific transformations from native data to a global common space using searchlight-based alignment.

**Parameters**

| Parameter | Default | Meaning |
| --- | --- | --- |
| `work_dir` | `required` | Root working directory containing subject data, common spaces, and output folders. |
| `sls` | `required` | Searchlight definitions. Each element is a list or array of vertex/voxel indices defining one searchlight. If a dict is provided, keys should correspond to structure names (e.g., 'hemi-L', 'hemi-R', 'l', 'r'). |
| `dists` | `required` | Distances between the searchlight center and its vertices/voxels. Required when `weight=True`. Keys must match those of `sls` if a dict is provided. |
| `radius` | `required` | Searchlight radius. For surface data, interpreted as geodesic distance (in mm); for volumetric data, interpreted as voxel steps. |
| `sub_list` | `required` | List of subject identifiers (e.g., ['sub-01', 'sub-02']). |
| `source_step` | `'resample'` | Name of the preprocessing step folder under each subject directory containing the source data to be aligned. |
| `modality` | `'response'` | Name of the modality folder under `source_step`. |
| `structure_name` | `['hemi-L', 'hemi-R']` | Structure(s) for which transformations are computed (e.g., 'hemi-L', 'hemi-R', or volumetric ROIs). |
| `task_name` | `'rha'` | Name identifying the target common space. This must match the `task-<task_name>` field used when generating the common space. |
| `cspace_desc` | `None` | Additional BIDS-style filters used to uniquely identify the target common space when multiple common spaces share the same `task_name` (e.g., {'space': 'surf'}). |
| `reflection` | `True` | Whether to allow reflections in the Procrustes transformation. |
| `scaling` | `False` | Whether to allow isotropic scaling in the Procrustes transformation. |
| `weight` | `True` | Whether to weight searchlight contributions according to the distance of vertices/voxels from the searchlight center when aggregating local transformations. |
| `chunk_size` | `2000` | Number of searchlights processed in one batch. Reduce this value if memory is limited. |
| `dtype` | `np.float32` | Data type used for the aggregated global transformation matrices. |
| `njobs` | `5` | Number of parallel workers. |
| `scope` | `'sls'` | Label used to define the output filename (default: 'sls'). |
| `suffix` | `None` | Optional suffix appended to generated output names. |
| `json_path` | `'prep_log.json'` | Path to the JSON file where processing parameters are recorded. |
| `overwrite` | `False` | Whether to overwrite existing entries in the JSON log. |
| `log_num` | `'00001'` | Identifier for the log file (e.g., '00001' → 'progress_00001.log'). |
| `**file_filters` | `optional` | Keyword arguments used to filter source data files following BIDS-style naming conventions (e.g., run='01'). |

**Returns or outputs**

None Subject-specific global transformation matrices are saved as `.npz` files under the corresponding subject directories.

### `xform_sls_ucon`

**Signature**

```python
fmriha.ha.xforms.xform_sls_ucon(work_dir, sls, dists, radius, sub_list, source_step='resample', modality='response', structure_name=['hemi-L', 'hemi-R'], task_name='rha1', cspace_kind='procrustes', common_topography=False, demean=True, start_sub=0, save_cspace=False, reflection=True, scaling=False, xform_weight=True, chunk_size=2000, dtype=np.float32, njobs=5, scope='sls', suffix=None, json_path='prep_log.json', overwrite=False, log_num='00001', verbose=1, **file_filters)
```

**Purpose**

Compute unconcatenated searchlight-based transformations to a common space.

**Parameters**

| Parameter | Default | Meaning |
| --- | --- | --- |
| `work_dir` | `required` | Root working directory containing subject data and output folders. |
| `sls` | `required` | Searchlight definitions, distance information, radius, and subject list. These parameters have the same meaning as in `xform_sls_con`. |
| `dists` | `required` | Distances associated with searchlight members. |
| `radius` | `required` | Searchlight radius. |
| `sub_list` | `required` | Subject identifiers or subject folder names to process. |
| `source_step` | `'resample'` | Name of the preprocessing step folder under each subject directory containing the source data. |
| `modality` | `'response'` | Name of the modality folder under `source_step`. |
| `structure_name` | `['hemi-L', 'hemi-R']` | Structure(s) for which transformations are computed (e.g., hemispheres or volumetric ROIs). |
| `task_name` | `'rha1'` | Identifier used to label output files generated by this function. |
| `cspace_kind` | `'procrustes'` | Method used to estimate searchlight-level common spaces. |
| `common_topography` | `False` | Parameters controlling common-space estimation, with the same meaning as in `cspace_sls`. |
| `demean` | `True` | Whether to remove the mean before decomposition or alignment. |
| `start_sub` | `0` | Subject index used as the first subject in common-space construction. |
| `save_cspace` | `False` | Whether to save searchlight-level common spaces to disk. Disabling this option is recommended when the number of searchlights is large. |
| `reflection` | `True` | Whether to allow reflection or scaling in the Procrustes transformation. |
| `scaling` | `False` | Whether to allow a global scale factor in the transformation. |
| `xform_weight` | `True` | Whether to apply distance-based weighting when aggregating local transformations into a global matrix. |
| `chunk_size` | `2000` | Parameters controlling batching, data type, and parallelization. |
| `dtype` | `np.float32` | NumPy data type used for saved or computed arrays. |
| `njobs` | `5` | Number of parallel jobs. |
| `scope` | `'sls'` | Parameters controlling output naming and logging. |
| `suffix` | `None` | Optional suffix appended to generated output names. |
| `json_path` | `'prep_log.json'` | Path to the preprocessing or pipeline log JSON file. |
| `overwrite` | `False` | Whether an existing file may be replaced. |
| `log_num` | `'00001'` | Log identifier written with pipeline records. |
| `verbose` | `1` | Verbosity level. |
| `**file_filters` | `optional` | Keyword arguments used to filter source data files following BIDS-style naming conventions. |

**Returns or outputs**

None Subject-specific global transformation matrices are saved as `.npz` files under the corresponding subject directories.

### `xform_sls_con_mega`

**Signature**

```python
fmriha.ha.xforms.xform_sls_con_mega(work_dir, sls, dists, radius, sub_list, source_step='resample', modality='response', structure_name=['hemi-L', 'hemi-R'], task_name='rha1', cspace_desc=None, reflection=True, scaling=False, weight=True, chunk_size=2000, dtype=np.float32, njobs=5, scope='sls', suffix=None, json_path='prep_log.json', overwrite=False, log_num='00001', **file_filters)
```

**Purpose**

Compute constrained transformation matrices with sparse chunk-wise accumulation.

**Parameters**

| Parameter | Default | Meaning |
| --- | --- | --- |
| `work_dir` | `required` | Root working directory for the project or pipeline outputs. |
| `sls` | `required` | Searchlight definitions; each item contains the feature indices in one local neighborhood. |
| `dists` | `required` | Distances associated with searchlight members. |
| `radius` | `required` | Searchlight radius. |
| `sub_list` | `required` | Subject identifiers or subject folder names to process. |
| `source_step` | `'resample'` | Processing step containing the source data to be transformed. |
| `modality` | `'response'` | Data modality label, such as response or connectivity. |
| `structure_name` | `['hemi-L', 'hemi-R']` | Brain structure name or list of structures to process. |
| `task_name` | `'rha1'` | Task or analysis label used in generated output names and logs. |
| `cspace_desc` | `None` | Descriptor used to locate a precomputed common-space file. |
| `reflection` | `True` | Whether to allow reflection in the Procrustes transformation. |
| `scaling` | `False` | Whether to allow a global scale factor in the transformation. |
| `weight` | `True` | Whether to apply searchlight distance weights during aggregation. |
| `chunk_size` | `2000` | Number of vertices, voxels, or searchlights processed per chunk. |
| `dtype` | `np.float32` | NumPy data type used for saved or computed arrays. |
| `njobs` | `5` | Number of parallel jobs. |
| `scope` | `'sls'` | Output scope, commonly searchlight-level or whole-structure aggregation. |
| `suffix` | `None` | Optional suffix appended to generated output names. |
| `json_path` | `'prep_log.json'` | Path to the preprocessing or pipeline log JSON file. |
| `overwrite` | `False` | Whether an existing file may be replaced. |
| `log_num` | `'00001'` | Log identifier written with pipeline records. |
| `**file_filters` | `optional` | Selection filters used to choose matching files or records. |

**Returns or outputs**

See the function output or saved pipeline records for returned objects and generated files.

### `xform_sls_ucon_mega`

**Signature**

```python
fmriha.ha.xforms.xform_sls_ucon_mega(work_dir, sls, dists, radius, sub_list, source_step='resample', modality='response', structure_name=['hemi-L', 'hemi-R'], task_name='rha1', cspace_kind='procrustes', common_topography=False, demean=True, start_sub=0, save_cspace=False, reflection=True, scaling=False, xform_weight=True, chunk_size=2000, dtype=np.float32, njobs=5, scope='sls', suffix=None, json_path='prep_log.json', overwrite=False, log_num='00001', verbose=1, **file_filters)
```

**Purpose**

Compute unconcatenated transformation matrices with sparse chunk-wise accumulation.

**Parameters**

| Parameter | Default | Meaning |
| --- | --- | --- |
| `work_dir` | `required` | Root working directory for the project or pipeline outputs. |
| `sls` | `required` | Searchlight definitions; each item contains the feature indices in one local neighborhood. |
| `dists` | `required` | Distances associated with searchlight members. |
| `radius` | `required` | Searchlight radius. |
| `sub_list` | `required` | Subject identifiers or subject folder names to process. |
| `source_step` | `'resample'` | Processing step containing the source data to be transformed. |
| `modality` | `'response'` | Data modality label, such as response or connectivity. |
| `structure_name` | `['hemi-L', 'hemi-R']` | Brain structure name or list of structures to process. |
| `task_name` | `'rha1'` | Task or analysis label used in generated output names and logs. |
| `cspace_kind` | `'procrustes'` | Common-space construction method, such as Procrustes or PCA. |
| `common_topography` | `False` | Whether to preserve a shared topographic basis during template construction. |
| `demean` | `True` | Whether to remove the mean before decomposition or alignment. |
| `start_sub` | `0` | Subject index used as the first subject in common-space construction. |
| `save_cspace` | `False` | Whether to save intermediate common-space estimates. |
| `reflection` | `True` | Whether to allow reflection in the Procrustes transformation. |
| `scaling` | `False` | Whether to allow a global scale factor in the transformation. |
| `xform_weight` | `True` | Whether to weight searchlight transformation matrices during aggregation. |
| `chunk_size` | `2000` | Number of vertices, voxels, or searchlights processed per chunk. |
| `dtype` | `np.float32` | NumPy data type used for saved or computed arrays. |
| `njobs` | `5` | Number of parallel jobs. |
| `scope` | `'sls'` | Output scope, commonly searchlight-level or whole-structure aggregation. |
| `suffix` | `None` | Optional suffix appended to generated output names. |
| `json_path` | `'prep_log.json'` | Path to the preprocessing or pipeline log JSON file. |
| `overwrite` | `False` | Whether an existing file may be replaced. |
| `log_num` | `'00001'` | Log identifier written with pipeline records. |
| `verbose` | `1` | Verbosity level. |
| `**file_filters` | `optional` | Selection filters used to choose matching files or records. |

**Returns or outputs**

See the function output or saved pipeline records for returned objects and generated files.

### `xform_sls_ucon_chunks`

**Signature**

```python
fmriha.ha.xforms.xform_sls_ucon_chunks(work_dir, sls, dists, radius, sub_list, source_step='resample', modality='response', structure_name=['hemi-L', 'hemi-R'], task_name='rha1', cspace_kind='procrustes', reflection=True, scaling=False, xform_weight=True, chunk_size=2000, dtype=np.float32, njobs=5, scope='sls', suffix=None, json_path='prep_log.json', overwrite=False, log_num='00001', **file_filters)
```

**Purpose**

Reconstruct unconcatenated transformation matrices from chunked searchlight common spaces.

**Parameters**

| Parameter | Default | Meaning |
| --- | --- | --- |
| `work_dir` | `required` | Root working directory for the project or pipeline outputs. |
| `sls` | `required` | Searchlight definitions; each item contains the feature indices in one local neighborhood. |
| `dists` | `required` | Distances associated with searchlight members. |
| `radius` | `required` | Searchlight radius. |
| `sub_list` | `required` | Subject identifiers or subject folder names to process. |
| `source_step` | `'resample'` | Processing step containing the source data to be transformed. |
| `modality` | `'response'` | Data modality label, such as response or connectivity. |
| `structure_name` | `['hemi-L', 'hemi-R']` | Brain structure name or list of structures to process. |
| `task_name` | `'rha1'` | Task or analysis label used in generated output names and logs. |
| `cspace_kind` | `'procrustes'` | Common-space construction method, such as Procrustes or PCA. |
| `reflection` | `True` | Whether to allow reflection in the Procrustes transformation. |
| `scaling` | `False` | Whether to allow a global scale factor in the transformation. |
| `xform_weight` | `True` | Whether to weight searchlight transformation matrices during aggregation. |
| `chunk_size` | `2000` | Number of vertices, voxels, or searchlights processed per chunk. |
| `dtype` | `np.float32` | NumPy data type used for saved or computed arrays. |
| `njobs` | `5` | Number of parallel jobs. |
| `scope` | `'sls'` | Output scope, commonly searchlight-level or whole-structure aggregation. |
| `suffix` | `None` | Optional suffix appended to generated output names. |
| `json_path` | `'prep_log.json'` | Path to the preprocessing or pipeline log JSON file. |
| `overwrite` | `False` | Whether an existing file may be replaced. |
| `log_num` | `'00001'` | Log identifier written with pipeline records. |
| `**file_filters` | `optional` | Selection filters used to choose matching files or records. |

**Returns or outputs**

See the function output or saved pipeline records for returned objects and generated files.

### `xform_sls_dask`

**Signature**

```python
fmriha.ha.xforms.xform_sls_dask(work_dir, sls, dists, radius, sub_list, source_step='resample', modality='response', structure_name=['hemi-L', 'hemi-R'], task_name='rha1', role='train', cspace_desc=None, reflection=True, scaling=False, weight=True, chunk_size=2000, dtype=np.float32, scope='sls', suffix=None, json_path='prep_log.json', overwrite=False, log_num='00001', use_local_cspace=False, **file_filters)
```

**Purpose**

Compute searchlight-based transformation matrices with Dask.

**Parameters**

| Parameter | Default | Meaning |
| --- | --- | --- |
| `work_dir` | `required` | Root working directory for the project or pipeline outputs. |
| `sls` | `required` | Searchlight definitions; each item contains the feature indices in one local neighborhood. |
| `dists` | `required` | Distances associated with searchlight members. |
| `radius` | `required` | Searchlight radius. |
| `sub_list` | `required` | Subject identifiers or subject folder names to process. |
| `source_step` | `'resample'` | Processing step containing the source data to be transformed. |
| `modality` | `'response'` | Data modality label, such as response or connectivity. |
| `structure_name` | `['hemi-L', 'hemi-R']` | Brain structure name or list of structures to process. |
| `task_name` | `'rha1'` | Task or analysis label used in generated output names and logs. |
| `role` | `'train'` | Analysis role label used when selecting or saving transformation outputs. |
| `cspace_desc` | `None` | Descriptor used to locate a precomputed common-space file. |
| `reflection` | `True` | Whether to allow reflection in the Procrustes transformation. |
| `scaling` | `False` | Whether to allow a global scale factor in the transformation. |
| `weight` | `True` | Whether to apply searchlight distance weights during aggregation. |
| `chunk_size` | `2000` | Number of vertices, voxels, or searchlights processed per chunk. |
| `dtype` | `np.float32` | NumPy data type used for saved or computed arrays. |
| `scope` | `'sls'` | Output scope, commonly searchlight-level or whole-structure aggregation. |
| `suffix` | `None` | Optional suffix appended to generated output names. |
| `json_path` | `'prep_log.json'` | Path to the preprocessing or pipeline log JSON file. |
| `overwrite` | `False` | Whether an existing file may be replaced. |
| `log_num` | `'00001'` | Log identifier written with pipeline records. |
| `use_local_cspace` | `False` | Whether to read local searchlight common spaces during Dask transformation estimation. |
| `**file_filters` | `optional` | Selection filters used to choose matching files or records. |

**Returns or outputs**

See the function output or saved pipeline records for returned objects and generated files.


### `xform_sls`

**Signature**

```python
fmriha.ha.xforms.xform_sls(work_dir, sls, dists, radius, sub_list, source_step='resample', modality='response', structure_name=['hemi-L', 'hemi-R'], task_name='rha1', cspace_kind='procrustes', cspace_desc=None, concatenated=True, common_topography=False, demean=True, start_sub=0, reflection=True, scaling=False, xform_weight=True, chunk_size=2000, dtype=np.float32, njobs=5, scope='sls', suffix=None, json_path='prep_log.json', overwrite=False, log_num='00001', verbose=1, dask=False, cluster=None, **file_filters)
```

**Purpose**

Dispatch searchlight transformation estimation based on common-space form.

**Parameters**

| Parameter | Default | Meaning |
| --- | --- | --- |
| `work_dir` | `required` | Root working directory for the project or pipeline outputs. |
| `sls` | `required` | Searchlight definitions; each item contains the feature indices in one local neighborhood. |
| `dists` | `required` | Distances associated with searchlight members. |
| `radius` | `required` | Searchlight radius. |
| `sub_list` | `required` | Subject identifiers or subject folder names to process. |
| `source_step` | `'resample'` | Processing step containing the source data to be transformed. |
| `modality` | `'response'` | Data modality label, such as response or connectivity. |
| `structure_name` | `['hemi-L', 'hemi-R']` | Brain structure name or list of structures to process. |
| `task_name` | `'rha1'` | Task or analysis label used in generated output names and logs. |
| `cspace_kind` | `'procrustes'` | Common-space construction method, such as Procrustes or PCA. |
| `cspace_desc` | `None` | Descriptor used to locate a precomputed common-space file. |
| `concatenated` | `True` | Select whether to use concatenated common space. - `True` calls `xform_sls_con`. - `False` first looks for chunked searchlight-level common spaces generated by `cspace_ucon_sls`. If none are found, it calls `xform_sls_ucon` with `save_cspace=False`. |
| `common_topography` | `False` | Whether to preserve a shared topographic basis during template construction. |
| `demean` | `True` | Whether to remove the mean before decomposition or alignment. |
| `start_sub` | `0` | Subject index used as the first subject in common-space construction. |
| `reflection` | `True` | Whether to allow reflection in the Procrustes transformation. |
| `scaling` | `False` | Whether to allow a global scale factor in the transformation. |
| `xform_weight` | `True` | Whether to weight searchlight transformation matrices during aggregation. |
| `chunk_size` | `2000` | Number of vertices, voxels, or searchlights processed per chunk. |
| `dtype` | `np.float32` | NumPy data type used for saved or computed arrays. |
| `njobs` | `5` | Number of parallel jobs. |
| `scope` | `'sls'` | Output scope, commonly searchlight-level or whole-structure aggregation. |
| `suffix` | `None` | Optional suffix appended to generated output names. |
| `json_path` | `'prep_log.json'` | Path to the preprocessing or pipeline log JSON file. |
| `overwrite` | `False` | Whether an existing file may be replaced. |
| `log_num` | `'00001'` | Log identifier written with pipeline records. |
| `verbose` | `1` | Verbosity level. |
| `dask` | `False` | Whether to use Dask-backed execution. |
| `cluster` | `None` | Dask cluster or cluster-like object used for distributed execution. |
| `**file_filters` | `optional` | Selection filters used to choose matching files or records. |

**Returns or outputs**

See the function output or saved pipeline records for returned objects and generated files.


## Apply Transformations

`fmriha.ha.align`

Application of precomputed transformations to subject data.


### `align_pipe`

**Signature**

```python
fmriha.ha.align.align_pipe(work_dir, sub_list, source_step, source_modality, source_structure_name, source_name_filter, xform_modality, xform_structure_name, xform_name_filter, njobs=5, verbose=1, dtype=np.float32, json_path='prep_log.json', overwrite=False, log_num='00001', dask=False, batch_size_dask=30, suffix=None)
```

**Purpose**

Apply precomputed transformation matrices to subject data in parallel.

**Parameters**

| Parameter | Default | Meaning |
| --- | --- | --- |
| `work_dir` | `required` | Root working directory containing subject data, transformation matrices, and output folders. |
| `sub_list` | `required` | Subject identifiers to align. |
| `source_step` | `required` | Parameters used to locate the source data files. |
| `source_modality` | `required` | Modality label for the source data. |
| `source_structure_name` | `required` | Structure name for the source data. |
| `source_name_filter` | `required` | BIDS-style filter used to locate the source data file for each subject. |
| `xform_modality` | `required` | Parameters used to locate the transform files. |
| `xform_structure_name` | `required` | Structure name used to select the transformation matrix. |
| `xform_name_filter` | `required` | BIDS-style filter used to locate the transformation matrix file for each subject. |
| `njobs` | `5` | Number of parallel workers. |
| `verbose` | `1` | Joblib verbosity level. |
| `dtype` | `np.float32` | Data type of the aligned output. |
| `json_path` | `'prep_log.json'` | JSON log path. |
| `overwrite` | `False` | Whether to overwrite existing entries in the JSON log. |
| `log_num` | `'00001'` | Identifier for the progress log file. |
| `dask` | `False` | Whether to use the current Dask client for alignment. |
| `batch_size_dask` | `30` | Number of subjects submitted in each Dask batch. |
| `suffix` | `None` | Optional suffix appended to the output filename. |

**Returns or outputs**

None Aligned data are saved as `.npy` files under each subject's `align` directory.


## Functional Connectivity

`fmriha.ha.fc`

Searchlight-level time-series averaging and connectivity construction.


### `sls_ts_mean`

**Signature**

```python
fmriha.ha.fc.sls_ts_mean(sub_data, sls, dtype=np.float32)
```

**Purpose**

Calculate mean time series for each searchlight region.

**Parameters**

| Parameter | Default | Meaning |
| --- | --- | --- |
| `sub_data` | `required` | Subject-level response matrix with shape (n_time, n_vertex). |
| `sls` | `required` | Searchlight definitions. Each item can contain one or more vertex/voxel indices. A scalar item is treated as a one-vertex searchlight. |
| `dtype` | `np.float32` | Data precision used for the returned array. |

**Returns or outputs**

sls_mean_series : ndarray Mean time series of each searchlight region, shape (n_time, n_searchlights).

### `fc_build`

**Signature**

```python
fmriha.ha.fc.fc_build(work_dir, sub_list, seed_step, seed_modality, seed_structure, seed_file_filter, seed_sls, target_step, target_modality, target_structure, target_file_filter, target_sls, zscore=True, njobs=5, verbose=1, dtype=np.float32, json_path='prep_log.json', overwrite=False, log_num='00001', suffix=None, get_name=False, dask=False, batch_size_dask=30)
```

**Purpose**

Compute subject-level functional connectivity matrices.

**Parameters**

| Parameter | Default | Meaning |
| --- | --- | --- |
| `work_dir` | `required` | Root working directory containing subject data and output folders. |
| `sub_list` | `required` | Subject identifiers to process. |
| `seed_step` | `required` | Parameters used to locate seed response files. |
| `seed_modality` | `required` | Modality label for seed-side data. |
| `seed_structure` | `required` | Seed structure folder. Exactly one structure is allowed. |
| `seed_file_filter` | `required` | Filter used to select seed-side files. |
| `seed_sls` | `required` | Seed searchlight definitions. |
| `target_step` | `required` | Parameters used to locate target response files. |
| `target_modality` | `required` | Modality label for the target data. |
| `target_structure` | `required` | Target structure folder or folders. Multiple structures are combined. |
| `target_file_filter` | `required` | Filter used to select target-side files. |
| `target_sls` | `required` | Target searchlight definitions. |
| `zscore` | `True` | If True, z-score FC values across target nodes for each seed. |
| `njobs` | `5` | Number of parallel jobs. |
| `verbose` | `1` | Joblib verbosity. |
| `dtype` | `np.float32` | Data precision used for intermediate and saved FC matrices. |
| `json_path` | `'prep_log.json'` | JSON log path. |
| `overwrite` | `False` | Whether to overwrite the JSON log. |
| `log_num` | `'00001'` | Progress log identifier. |
| `suffix` | `None` | Optional suffix appended to output filenames. |
| `get_name` | `False` | If True, return saved FC filenames. |
| `dask` | `False` | Whether to use the current Dask client for FC construction. |
| `batch_size_dask` | `30` | Number of subjects submitted in each Dask batch. |

**Returns or outputs**

None or list of Path Saved FC filenames when `get_name=True`; otherwise None.


## Distributed Helpers

`fmriha.ha.worker_funcs`

Dask helpers for broadcasting data and managing worker clients.

These helpers are intended for distributed execution environments.


### `class BroadcastPlugin`

**Purpose**

Dask worker plugin that attaches a shared NumPy array to each worker.

**Public methods**

### `BroadcastPlugin.__init__`

**Signature**

```python
fmriha.ha.worker_funcs.BroadcastPlugin.__init__(data)
```

**Purpose**

Performs the operation indicated by the function name and arguments.

**Parameters**

| Parameter | Default | Meaning |
| --- | --- | --- |
| `data` | `required` | Input array or matrix. |

**Returns or outputs**

See the function output or saved pipeline records for returned objects and generated files.

### `resilient_client`

**Signature**

```python
fmriha.ha.worker_funcs.resilient_client(cluster, timeout=1200, max_retries=3)
```

**Purpose**

Create a Dask `Client` with retry logic.

**Parameters**

| Parameter | Default | Meaning |
| --- | --- | --- |
| `cluster` | `required` | A Dask cluster object accepted by :class:`dask.distributed.Client`. For example, a `LocalCluster` or an adaptive/remote cluster handle. |
| `timeout` | `1200` | Connection timeout in seconds passed to the `Client` constructor. Default is 1200. |
| `max_retries` | `3` | Maximum number of connection attempts. Default is 3. |

**Returns or outputs**

client : dask.distributed.Client A connected Dask client.

### `disable_reduce_replicas_on_scheduler`

**Signature**

```python
fmriha.ha.worker_funcs.disable_reduce_replicas_on_scheduler(dask_scheduler)
```

**Purpose**

Disable the Active Memory Manager `ReduceReplicas` policy.

**Parameters**

| Parameter | Default | Meaning |
| --- | --- | --- |
| `dask_scheduler` | `required` | The scheduler instance whose AMM policies will be modified. |

**Returns or outputs**

None

### `reset_plugin`

**Signature**

```python
fmriha.ha.worker_funcs.reset_plugin(cluster, sub_file_list)
```

**Purpose**

Load multiple NumPy arrays, stack them, and broadcast to workers via a plugin.

**Parameters**

| Parameter | Default | Meaning |
| --- | --- | --- |
| `cluster` | `required` | A Dask cluster object accepted by :class:`dask.distributed.Client`. |
| `sub_file_list` | `required` | Paths to `.npy` files. Each file should contain an array with the same shape and dtype so that :func:`numpy.stack` can combine them. |

**Returns or outputs**

client : dask.distributed.Client A connected client with the `tmpl_plugin` registered.


## Pipeline Script Builder

`fmriha.pipeline._builder`

Shared helper functions for JSON-safe configuration and script rendering.

These helpers are useful when extending script generation or serializing pipeline configurations.


### `code`

**Signature**

```python
fmriha.pipeline._builder.code(expr)
```

**Purpose**

Mark one config value as raw Python code in the generated script.

**Parameters**

| Parameter | Default | Meaning |
| --- | --- | --- |
| `expr` | `required` | Python expression to be emitted directly into the rendered script. |

**Returns or outputs**

dict Marker dictionary for deferred code rendering.

### `is_code_marker`

**Signature**

```python
fmriha.pipeline._builder.is_code_marker(value)
```

**Purpose**

Return True when a value is a raw-code marker dictionary.

**Parameters**

| Parameter | Default | Meaning |
| --- | --- | --- |
| `value` | `required` | Value to serialize, restore, or test for a code marker. |

**Returns or outputs**

See the function output or saved pipeline records for returned objects and generated files.

### `normalize_for_json`

**Signature**

```python
fmriha.pipeline._builder.normalize_for_json(value)
```

**Purpose**

Convert config objects into JSON-safe values.

**Parameters**

| Parameter | Default | Meaning |
| --- | --- | --- |
| `value` | `required` | Config object or nested config field. |

**Returns or outputs**

any JSON-safe value with paths stringified and code markers preserved.

### `restore_from_json`

**Signature**

```python
fmriha.pipeline._builder.restore_from_json(value)
```

**Purpose**

Restore config objects after JSON loading.

**Parameters**

| Parameter | Default | Meaning |
| --- | --- | --- |
| `value` | `required` | JSON-loaded config object or nested field. |

**Returns or outputs**

any Restored config value with code markers preserved.

### `python_literal`

**Signature**

```python
fmriha.pipeline._builder.python_literal(value)
```

**Purpose**

Render one config value as Python source code.

**Parameters**

| Parameter | Default | Meaning |
| --- | --- | --- |
| `value` | `required` | Config value to render. |

**Returns or outputs**

str Python source string for the given value.

### `render_header`

**Signature**

```python
fmriha.pipeline._builder.render_header(config, created_with)
```

**Purpose**

Render the header block of the generated script.

**Parameters**

| Parameter | Default | Meaning |
| --- | --- | --- |
| `config` | `required` | Pipeline configuration. |
| `created_with` | `required` | Module name used in the generated docstring footer. |

**Returns or outputs**

str Header text including shebang, module docstring, imports, and preamble.

### `render_shared_block`

**Signature**

```python
fmriha.pipeline._builder.render_shared_block(config)
```

**Purpose**

Render the shared-parameter block of the generated script.

**Parameters**

| Parameter | Default | Meaning |
| --- | --- | --- |
| `config` | `required` | Pipeline configuration. |

**Returns or outputs**

str Python code defining the shared parameter dictionary.

### `save_config_json`

**Signature**

```python
fmriha.pipeline._builder.save_config_json(config_path, config, overwrite=True)
```

**Purpose**

Save one config dictionary to local JSON.

**Parameters**

| Parameter | Default | Meaning |
| --- | --- | --- |
| `config_path` | `required` | Output JSON path. |
| `config` | `required` | Config dictionary to save. |
| `overwrite` | `True` | Whether to overwrite an existing file. |

**Returns or outputs**

Path Saved config path.

### `load_config_json`

**Signature**

```python
fmriha.pipeline._builder.load_config_json(config_path)
```

**Purpose**

Load one config dictionary from local JSON.

**Parameters**

| Parameter | Default | Meaning |
| --- | --- | --- |
| `config_path` | `required` | Path to a saved JSON config. |

**Returns or outputs**

dict Loaded config dictionary.


## Preprocessing Pipeline Builder

`fmriha.pipeline.preprocessing_pipeline`

Configuration and script generation for preprocessing workflows.


### `default_preprocessing_config`

**Signature**

```python
fmriha.pipeline.preprocessing_pipeline.default_preprocessing_config()
```

**Purpose**

Build a starter config for the preprocessing workflow.

**Parameters**

This function takes no parameters.

**Returns or outputs**

dict Pipeline configuration containing one ordered step list.

### `save_preprocessing_config`

**Signature**

```python
fmriha.pipeline.preprocessing_pipeline.save_preprocessing_config(config_path, config, overwrite=True, log_json_path=None)
```

**Purpose**

Save one preprocessing pipeline config to local JSON.

**Parameters**

| Parameter | Default | Meaning |
| --- | --- | --- |
| `config_path` | `required` | Output JSON path for the pipeline config. |
| `config` | `required` | Pipeline config returned by `default_preprocessing_config` or a compatible custom config. |
| `overwrite` | `True` | Whether to overwrite an existing config file. |
| `log_json_path` | `None` | Optional JSON log path used to record the save event. |

**Returns or outputs**

See the function output or saved pipeline records for returned objects and generated files.

### `load_preprocessing_config`

**Signature**

```python
fmriha.pipeline.preprocessing_pipeline.load_preprocessing_config(config_path)
```

**Purpose**

Load a previously saved preprocessing pipeline config JSON.

**Parameters**

| Parameter | Default | Meaning |
| --- | --- | --- |
| `config_path` | `required` | Path to a saved preprocessing pipeline config. |

**Returns or outputs**

See the function output or saved pipeline records for returned objects and generated files.

### `generate_preprocessing_script`

**Signature**

```python
fmriha.pipeline.preprocessing_pipeline.generate_preprocessing_script(config)
```

**Purpose**

Render a standalone Python script from one preprocessing config.

**Parameters**

| Parameter | Default | Meaning |
| --- | --- | --- |
| `config` | `required` | Preprocessing pipeline configuration. |

**Returns or outputs**

str Generated Python script text.

### `write_preprocessing_script`

**Signature**

```python
fmriha.pipeline.preprocessing_pipeline.write_preprocessing_script(output_path, config, overwrite=True, config_path=None, save_config=False, log_json_path=None)
```

**Purpose**

Write one generated preprocessing script to disk.

**Parameters**

| Parameter | Default | Meaning |
| --- | --- | --- |
| `output_path` | `required` | Destination `.py` file. |
| `config` | `required` | Preprocessing pipeline configuration. |
| `overwrite` | `True` | Whether to overwrite an existing script. |
| `config_path` | `None` | Optional config output path. Used only when `save_config=True`. |
| `save_config` | `False` | If True, also save the config to `config_path`. |
| `log_json_path` | `None` | Optional JSON log path used to record generation events. |

**Returns or outputs**

See the function output or saved pipeline records for returned objects and generated files.


## Hyperalignment Pipeline Builder

`fmriha.pipeline.ha_pipeline`

Configuration and script generation for response and connectivity hyperalignment workflows.


### `default_hyperalignment_config`

**Signature**

```python
fmriha.pipeline.ha_pipeline.default_hyperalignment_config(alignment_kind='response')
```

**Purpose**

Build a starter config for response or connectivity hyperalignment.

**Parameters**

| Parameter | Default | Meaning |
| --- | --- | --- |
| `alignment_kind` | `'response'` | Pipeline preset to generate. |

**Returns or outputs**

dict Pipeline configuration with shared parameters and an ordered step list. The default HA script uses `ha.cspace_sls` for common-space generation and `ha.xform_sls` for transformation estimation. Edit `shared['save_local_cspace']` to save searchlight-level common-space chunks, and edit `shared['xform_concatenated']` to select concatenated or unconcatenated transformation dispatch.

### `save_pipeline_config`

**Signature**

```python
fmriha.pipeline.ha_pipeline.save_pipeline_config(config_path, config, overwrite=True, log_json_path=None)
```

**Purpose**

Save one HA pipeline config to local JSON.

**Parameters**

| Parameter | Default | Meaning |
| --- | --- | --- |
| `config_path` | `required` | Output JSON path for the pipeline config. |
| `config` | `required` | Pipeline config returned by `default_hyperalignment_config` or a compatible custom config. |
| `overwrite` | `True` | Whether to overwrite an existing config file. |
| `log_json_path` | `None` | Optional JSON log path used to record the save event. |

**Returns or outputs**

See the function output or saved pipeline records for returned objects and generated files.

### `load_pipeline_config`

**Signature**

```python
fmriha.pipeline.ha_pipeline.load_pipeline_config(config_path)
```

**Purpose**

Load a previously saved HA pipeline config JSON.

**Parameters**

| Parameter | Default | Meaning |
| --- | --- | --- |
| `config_path` | `required` | Path to a saved pipeline config. |

**Returns or outputs**

See the function output or saved pipeline records for returned objects and generated files.

### `generate_hyperalignment_script`

**Signature**

```python
fmriha.pipeline.ha_pipeline.generate_hyperalignment_script(config)
```

**Purpose**

Render a standalone Python script from one HA pipeline config.

**Parameters**

| Parameter | Default | Meaning |
| --- | --- | --- |
| `config` | `required` | Pipeline configuration. |

**Returns or outputs**

str Generated Python script text.

### `write_hyperalignment_script`

**Signature**

```python
fmriha.pipeline.ha_pipeline.write_hyperalignment_script(output_path, config, overwrite=True, config_path=None, save_config=False, log_json_path=None)
```

**Purpose**

Write one generated HA script to disk.

**Parameters**

| Parameter | Default | Meaning |
| --- | --- | --- |
| `output_path` | `required` | Destination `.py` file. |
| `config` | `required` | Pipeline configuration. |
| `overwrite` | `True` | Whether to overwrite an existing script. |
| `config_path` | `None` | Optional config output path. Used only when `save_config=True`. |
| `save_config` | `False` | If True, also save the config to `config_path`. |
| `log_json_path` | `None` | Optional JSON log path used to record generation events. |

**Returns or outputs**

See the function output or saved pipeline records for returned objects and generated files.


## GLM and Background Connectivity

`fmriha.stat.glm`

Task-design modeling, beta extraction, residual time series, and background connectivity.


### `load_events_tsv`

**Signature**

```python
fmriha.stat.glm.load_events_tsv(events_tsv)
```

**Purpose**

Load an events TSV file and validate required event columns.

**Parameters**

| Parameter | Default | Meaning |
| --- | --- | --- |
| `events_tsv` | `required` | Path to an events TSV file with at least `onset`, `duration`, and `trial_type` columns. |

**Returns or outputs**

pandas.DataFrame Validated events table.

### `safe_zscore`

**Signature**

```python
fmriha.stat.glm.safe_zscore(data, axis=0, ddof=0, dtype=np.float64)
```

**Purpose**

Z-score an array while safely handling constant columns or rows.

**Parameters**

| Parameter | Default | Meaning |
| --- | --- | --- |
| `data` | `required` | Input array or matrix. |
| `axis` | `0` | Axis along which the operation is applied. |
| `ddof` | `0` | Delta degrees of freedom used in standard-deviation calculations. |
| `dtype` | `np.float64` | NumPy data type used for saved or computed arrays. |

**Returns or outputs**

See the function output or saved pipeline records for returned objects and generated files.

### `build_design_matrix_from_tsv`

**Signature**

```python
fmriha.stat.glm.build_design_matrix_from_tsv(events_tsv, n_scans, t_r, hrf_model='spm + derivative', drift_model='cosine', high_pass=0.01, fir_delays=None, add_regs=None, add_reg_names=None, min_onset=-24)
```

**Purpose**

Build an HRF-convolved first-level design matrix from an events TSV file.

**Parameters**

| Parameter | Default | Meaning |
| --- | --- | --- |
| `events_tsv` | `required` | Events TSV with `onset`, `duration`, and `trial_type` columns. Onset and duration values should use seconds, matching Nilearn's `frame_times` convention. |
| `n_scans` | `required` | Number of fMRI time points. |
| `t_r` | `required` | Repetition time in seconds. |
| `hrf_model` | `'spm + derivative'` | HRF model passed to Nilearn. |
| `drift_model` | `'cosine'` | Drift model passed to Nilearn. |
| `high_pass` | `0.01` | High-pass frequency cutoff used by Nilearn when applicable. |
| `fir_delays` | `None` | FIR delays. Required when `hrf_model="fir"`. |
| `add_regs` | `None` | Optional confounds shaped `n_scans x n_confounds`. |
| `add_reg_names` | `None` | Optional names for confound columns. |
| `min_onset` | `-24` | Minimum onset passed to Nilearn. |

**Returns or outputs**

pandas.DataFrame Design matrix with named columns.

### `fit_glm_matrix`

**Signature**

```python
fmriha.stat.glm.fit_glm_matrix(data, design_matrix, noise_model='ols', standardize_data=False, return_predicted=True, return_residuals=True)
```

**Purpose**

Fit a Nilearn GLM to a time-by-feature data matrix.

**Parameters**

| Parameter | Default | Meaning |
| --- | --- | --- |
| `data` | `required` | fMRI matrix shaped `n_timepoints x n_features`. |
| `design_matrix` | `required` | Design matrix shaped `n_timepoints x n_regressors`. |
| `noise_model` | `'ols'` | Noise model passed to `nilearn.glm.first_level.run_glm`. |
| `standardize_data` | `False` | If True, z-score each feature along the time axis before fitting. |
| `return_predicted` | `True` | If True, include the fitted time series in the returned dictionary. |
| `return_residuals` | `True` | If True, include residual time series in the returned dictionary. |

**Returns or outputs**

dict Structured GLM result with `labels`, `results`, `design_matrix`, `design_columns`, `beta`, `predicted`, `residuals`, and `r_square`.

### `extract_condition_betas`

**Signature**

```python
fmriha.stat.glm.extract_condition_betas(beta, design_columns, condition_names, include_derivatives=False)
```

**Purpose**

Extract condition-level beta vectors from a full beta matrix.

**Parameters**

| Parameter | Default | Meaning |
| --- | --- | --- |
| `beta` | `required` | Beta matrix shaped `n_regressors x n_features`. |
| `design_columns` | `required` | Column names from the design matrix. |
| `condition_names` | `required` | Condition names to extract. |
| `include_derivatives` | `False` | If True, also return derivative and FIR-delay beta vectors with clear keys such as `condition_derivative` or `condition_delay_1`. |

**Returns or outputs**

dict Mapping from condition/regressor name to beta vector. The pipeline function `glm_pipe` saves these vectors as `1 x n_features` arrays.

### `get_residual_timeseries`

**Signature**

```python
fmriha.stat.glm.get_residual_timeseries(glm_result=None, data=None, design_matrix=None, **fit_kwargs)
```

**Purpose**

Return residual time series from an existing result or a new GLM fit.

**Parameters**

| Parameter | Default | Meaning |
| --- | --- | --- |
| `glm_result` | `None` | Existing GLM result dictionary used to retrieve residuals. |
| `data` | `None` | Input array or matrix. |
| `design_matrix` | `None` | Design matrix used for GLM fitting. |
| `**fit_kwargs` | `optional` | Additional keyword arguments passed to GLM fitting. |

**Returns or outputs**

See the function output or saved pipeline records for returned objects and generated files.

### `compute_background_fc`

**Signature**

```python
fmriha.stat.glm.compute_background_fc(residuals, method='correlation', fisher_z=False, standardize=True)
```

**Purpose**

Compute background functional connectivity from residual time series.

**Parameters**

| Parameter | Default | Meaning |
| --- | --- | --- |
| `residuals` | `required` | Residual matrix shaped `n_timepoints x n_regions`. Inputs are assumed to already be ROI-level signals. |
| `method` | `'correlation'` | Connectivity metric. Only Pearson correlation is currently supported. |
| `fisher_z` | `False` | If True, apply Fisher's r-to-z transform to off-diagonal correlations. |
| `standardize` | `True` | If True, z-score each region before computing correlations. |

**Returns or outputs**

numpy.ndarray Functional connectivity matrix shaped `n_regions x n_regions`.

### `glm_pipe`

**Signature**

```python
fmriha.stat.glm.glm_pipe(work_dir, sub_list, events_tsv, input_pattern=None, output_types=('beta', 'residuals'), condition_names=None, t_r=None, hrf_model='spm + derivative', drift_model='cosine', high_pass=0.01, fir_delays=None, step=None, modality=None, structure_name=None, file_filter=None, confounds_pattern=None, add_reg_names=None, noise_model='ols', standardize_data=False, include_derivatives=False, suffix=None, n_jobs=1, verbose=1, overwrite=False, log_num='00004')
```

**Purpose**

Run subject-level matrix GLM on `.npy` time-by-feature matrices.

**Parameters**

| Parameter | Default | Meaning |
| --- | --- | --- |
| `work_dir` | `required` | Root working directory. |
| `sub_list` | `required` | Subject folder names, for example `["sub-01", "sub-02"]`. |
| `events_tsv` | `required` | Events TSV file(s) with `onset`, `duration`, and `trial_type` columns. A single path or one-item list is shared by all subjects. A list with the same length as `sub_list` is matched to subjects by order. |
| `input_pattern` | `None` | Relative input path template with a `{sub}` placeholder. If provided, this direct pattern is used to locate each subject input file. |
| `output_types` | `('beta', 'residuals')` | Requested outputs. |
| `condition_names` | `None` | Conditions to save as beta vectors. If None, conditions are inferred from `trial_type` in the events TSV. |
| `t_r` | `None` | Repetition time. This must be provided explicitly. |
| `hrf_model` | `'spm + derivative'` | Design matrix parameters passed to `build_design_matrix_from_tsv`. |
| `drift_model` | `'cosine'` | Drift model used in first-level GLM design construction. |
| `high_pass` | `0.01` | High-pass cutoff used for drift regressors. |
| `fir_delays` | `None` | FIR delay indices when using an FIR HRF model. |
| `step` | `None` | Pipeline folder components used to locate files when `input_pattern=None`. The searched folder is `work_dir / sub / step / modality / structure_name`. |
| `modality` | `None` | Data modality label, such as response or connectivity. |
| `structure_name` | `None` | One or more structure folders to process when `input_pattern=None`. |
| `file_filter` | `None` | BIDS-style filename filter passed to `fio.bids_filter` when `input_pattern=None`. For example `{"ses": "raiders"}`. |
| `confounds_pattern` | `None` | Optional relative path template to a subject confounds file. Supported formats are `.npy`, `.tsv`, and `.csv`. |
| `add_reg_names` | `None` | Confound column names. For `.tsv`/`.csv`, these names are selected from the file. For `.npy`, they name the columns. |
| `noise_model` | `'ols'` | Noise model passed to `fit_glm_matrix`. |
| `standardize_data` | `False` | If True, z-score each feature across time before fitting. |
| `include_derivatives` | `False` | If True, save derivative or FIR-delay beta vectors with explicit names. |
| `suffix` | `None` | Optional filename suffix. If None, no suffix field is added. |
| `n_jobs` | `1` | Number of subject-level workers. |
| `verbose` | `1` | Verbosity level for prints and joblib. |
| `overwrite` | `False` | If False, a subject is skipped when planned output files already exist. |
| `log_num` | `'00004'` | Progress log identifier. |

**Returns or outputs**

pandas.DataFrame One row per subject and structure with status, output paths, and error messages. Outputs are saved under structure-specific folders, for example `sub-01/glm/residuals/hemi-L`. Example >>> summary = glm_pipe( ... work_dir="/path/to/work_dir", ... sub_list=["sub-01", "sub-02"], ... events_tsv="/path/to/events.tsv", ... step="resample", ... modality="response", ... structure_name=["hemi-L", "hemi-R"], ... file_filter={"ses": "raiders"}, ... output_types=("beta", "residuals"), ... condition_names=None, ... t_r=1.0, ... hrf_model="spm + derivative", ... suffix="movie", ... n_jobs=2, ... overwrite=False, ... )


## Inter-Subject Correlation

`fmriha.stat.isc`

Response, functional-connectivity, and representational-geometry ISC routines.


### `fisher_rmean`

**Signature**

```python
fmriha.stat.isc.fisher_rmean(rvals, axis=None)
```

**Purpose**

Compute the Fisher z mean of correlation values.

**Parameters**

| Parameter | Default | Meaning |
| --- | --- | --- |
| `rvals` | `required` | Correlation values. |
| `axis` | `None` | Axis along which the mean is computed. |

**Returns or outputs**

ndarray Fisher z averaged correlations.

### `isc_single`

**Signature**

```python
fmriha.stat.isc.isc_single(sub_data, mat_sum, metric='correlation')
```

**Purpose**

Compute leave-one-out ISC for one subject.

**Parameters**

| Parameter | Default | Meaning |
| --- | --- | --- |
| `sub_data` | `required` | Subject data matrix with shape `(n_time, n_vertex)`. |
| `mat_sum` | `required` | Sum across subjects with shape `(n_time, n_vertex)`. |
| `metric` | `'correlation'` | Similarity metric used to compare one subject with the leave-one-out group signal. |

**Returns or outputs**

ndarray ISC values with shape `(n_vertex,)`.

### `isc_mat_nb`

**Signature**

```python
fmriha.stat.isc.isc_mat_nb(mat, pairwise=False, metric='correlation', n_jobs=5, summary_statistic='mean')
```

**Purpose**

Compute ISC from a 3D data matrix with joblib parallelization.

**Parameters**

| Parameter | Default | Meaning |
| --- | --- | --- |
| `mat` | `required` | Data matrix with shape `(n_time, n_vertex, n_subject)`. |
| `pairwise` | `False` | If True, compute pairwise ISC for all subject pairs. If False, compute leave-one-out ISC. |
| `metric` | `'correlation'` | Similarity metric. For pairwise mode, this is passed to `pdist` and converted from distance to similarity using `1 - distance`. |
| `n_jobs` | `5` | Number of parallel workers. |
| `summary_statistic` | `'mean'` | Summary statistic saved alongside the raw ISC results. |

**Returns or outputs**

dict Dictionary containing the raw ISC matrix and the requested summary statistic.

### `response_isc_pipe`

**Signature**

```python
fmriha.stat.isc.response_isc_pipe(work_dir, sub_list, step, modality, structure_name, file_filter, pairwise=False, metric='correlation', njobs=5, chunk_size=2000, dtype=np.float32, summary_statistic='mean', json_path='prep_log.json', overwrite=False, log_num='00003', suffix=None, time_split=None, plot_config=None)
```

**Purpose**

Run response ISC on processed subject data saved in the HA pipeline layout.

**Parameters**

| Parameter | Default | Meaning |
| --- | --- | --- |
| `work_dir` | `required` | Root HA working directory. |
| `sub_list` | `required` | Subject folder names, such as `['sub-01', 'sub-02']`. |
| `step` | `required` | Pipeline step and modality used to locate the input files. |
| `modality` | `required` | Data modality label, such as response or connectivity. |
| `structure_name` | `required` | One or more structure folders to process. |
| `file_filter` | `required` | BIDS-style filter passed to `fio.bids_filter`. Files with a `split` tag are excluded automatically. |
| `pairwise` | `False` | If True, compute pairwise ISC. Otherwise compute leave-one-out ISC. |
| `metric` | `'correlation'` | Similarity metric supported by the legacy block ISC implementation. |
| `njobs` | `5` | Number of workers used in the large-sample parallel path. |
| `chunk_size` | `2000` | Number of vertices/voxels processed per chunk. |
| `dtype` | `np.float32` | Output dtype for computed ISC values. |
| `summary_statistic` | `'mean'` | Summary statistic saved alongside the raw ISC array. |
| `json_path` | `'prep_log.json'` | JSON log path. |
| `overwrite` | `False` | Whether to overwrite existing JSON log entries. |
| `log_num` | `'00003'` | Progress log identifier. |
| `suffix` | `None` | Optional suffix appended to output filenames. |
| `time_split` | `None` | Optional time segmentation strategy. |
| `plot_config` | `None` | Optional plotting configuration used after result pkl files are saved. `None` or `False` disables plotting and keeps the original pipeline behavior. `True` saves default histogram figures only. A dictionary gives explicit control. Common keys are: `method` : str Result field to plot, usually `'mean'` or `'median'`. Defaults to `summary_statistic`. `histogram` : bool or dict If True, save histogram figures to `figures` under the result folder. A dict may include arguments for `fmriha.plot.plot_result_histogram`, such as `bins` or `color`. `surface` : bool or dict If enabled, save surface maps for matched `hemi-L`/`hemi-R` result files. Surface plotting requires `lh` and `rh` surface inputs, either at the top level or inside `surface`. Optional keys include `medial_wall`, `color_range` and `cmap`. `plot_dir` : str or Path Custom figure directory. Defaults to `<result_folder>/figures`. `show` : bool Whether to display figures while saving. Defaults to False for pipeline use. `fail_on_error` : bool If False, plotting errors are reported as warnings and do not stop the statistical pipeline. Example:: plot_config = { "histogram": {"bins": 40}, "surface": { "enabled": True, "lh": lh, "rh": rh, "medial_wall": medial_wall, "color_range": (-0.2, 0.6), "cmap": "Reds" }, "show": False } |

**Returns or outputs**

See the function output or saved pipeline records for returned objects and generated files.

### `fc_isc_pipe`

**Signature**

```python
fmriha.stat.isc.fc_isc_pipe(work_dir, sub_list, step, modality, seed_structure, target_structure, file_filter, seed_sls, target_sls, zscore=True, njobs=5, verbose=1, pairwise=False, metric='correlation', summary_statistic='mean', chunk_size=2000, dtype=np.float32, json_path='prep_log.json', overwrite=False, log_num='00003', suffix=None, delete_fc=False, time_split=None, target_filter=None, plot_config=None)
```

**Purpose**

Run ISC on subject-level functional connectivity matrices.

**Parameters**

| Parameter | Default | Meaning |
| --- | --- | --- |
| `work_dir` | `required` | Root HA working directory. |
| `sub_list` | `required` | Subject folder names, such as `['sub-01', 'sub-02']`. |
| `step` | `required` | Pipeline step and modality used to locate the response inputs. |
| `modality` | `required` | Data modality label, such as response or connectivity. |
| `seed_structure` | `required` | Seed structure folder used to build connectivity. |
| `target_structure` | `required` | One or more target structure folders combined into the FC output. |
| `file_filter` | `required` | BIDS-style filter used to find seed response files. |
| `seed_sls` | `required` | Searchlight definitions for seed and target structures. |
| `target_sls` | `required` | Searchlights used to summarize target-side data. |
| `zscore` | `True` | Whether to z-score FC matrices after computation. |
| `njobs` | `5` | Number of parallel workers. |
| `verbose` | `1` | Joblib verbosity level. |
| `pairwise` | `False` | If True, compute pairwise ISC. Otherwise compute leave-one-out ISC. |
| `metric` | `'correlation'` | Similarity metric supported by the legacy block ISC implementation. |
| `summary_statistic` | `'mean'` | Summary statistic saved alongside the raw ISC matrix. |
| `chunk_size` | `2000` | Number of FC features processed per chunk. |
| `dtype` | `np.float32` | Output dtype for FC and ISC computations. |
| `json_path` | `'prep_log.json'` | JSON log path. |
| `overwrite` | `False` | Whether to overwrite existing JSON log entries. |
| `log_num` | `'00003'` | Progress log identifier. |
| `suffix` | `None` | Optional suffix appended to output filenames. |
| `delete_fc` | `False` | If True, remove intermediate FC files after ISC is saved. |
| `time_split` | `None` | Optional time segmentation strategy. |
| `target_filter` | `None` | Optional BIDS-style filter used for target response files. If None, `file_filter` is reused. |
| `plot_config` | `None` | Optional plotting configuration used after result pkl files are saved. `None` or `False` disables plotting and keeps the original pipeline behavior. `True` saves default histogram figures only. A dictionary gives explicit control. Common keys are: `method` : str Result field to plot, usually `'mean'` or `'median'`. Defaults to `summary_statistic`. `histogram` : bool or dict If True, save histogram figures to `figures` under the result folder. A dict may include arguments for `fmriha.plot.plot_result_histogram`, such as `bins` or `color`. `surface` : bool or dict If enabled, save surface maps for matched `hemi-L`/`hemi-R` result files. Surface plotting requires `lh` and `rh` surface inputs, either at the top level or inside `surface`. Optional keys include `medial_wall`, `color_range` and `cmap`. `plot_dir` : str or Path Custom figure directory. Defaults to `<result_folder>/figures`. `show` : bool Whether to display figures while saving. Defaults to False for pipeline use. `fail_on_error` : bool If False, plotting errors are reported as warnings and do not stop the statistical pipeline. Example:: plot_config = { "histogram": {"bins": 40}, "surface": { "enabled": True, "lh": lh, "rh": rh, "medial_wall": medial_wall, "color_range": (-0.2, 0.6), "cmap": "Reds" }, "show": False } |

**Returns or outputs**

See the function output or saved pipeline records for returned objects and generated files.

### `rg_isc_pipe`

**Signature**

```python
fmriha.stat.isc.rg_isc_pipe(work_dir, sub_list, step, modality, structure_name, sls, file_filter, pairwise=False, njobs=5, verbose=1, dtype=np.float32, summary_statistic='mean', json_path='prep_log.json', overwrite=False, log_num='00004', suffix=None, time_split=None, plot_config=None)
```

**Purpose**

Run ISC on representational-geometry features derived from response data.

**Parameters**

| Parameter | Default | Meaning |
| --- | --- | --- |
| `work_dir` | `required` | Root HA working directory. |
| `sub_list` | `required` | Subject folder names, such as `['sub-01', 'sub-02']`. |
| `step` | `required` | Pipeline step and modality used to locate the response inputs. |
| `modality` | `required` | Data modality label, such as response or connectivity. |
| `structure_name` | `required` | One or more structure folders to process. |
| `sls` | `required` | Searchlight definitions used to build RG vectors. |
| `file_filter` | `required` | BIDS-style filter passed to `fio.bids_filter`. Files with a `split` tag are excluded automatically. |
| `pairwise` | `False` | If True, compute pairwise RG-ISC. Otherwise compute leave-one-out RG-ISC. |
| `njobs` | `5` | Number of parallel workers. |
| `verbose` | `1` | Joblib verbosity level. |
| `dtype` | `np.float32` | Output dtype used for RG-ISC values. |
| `summary_statistic` | `'mean'` | Summary statistic saved alongside the raw RG-ISC matrix. |
| `json_path` | `'prep_log.json'` | JSON log path. |
| `overwrite` | `False` | Whether to overwrite existing JSON log entries. |
| `log_num` | `'00004'` | Progress log identifier. |
| `suffix` | `None` | Optional suffix appended to output filenames. |
| `time_split` | `None` | Optional time segmentation strategy. |
| `plot_config` | `None` | Optional plotting configuration used after result pkl files are saved. `None` or `False` disables plotting and keeps the original pipeline behavior. `True` saves default histogram figures only. A dictionary gives explicit control. Common keys are: `method` : str Result field to plot, usually `'mean'` or `'median'`. Defaults to `summary_statistic`. `histogram` : bool or dict If True, save histogram figures to `figures` under the result folder. A dict may include arguments for `fmriha.plot.plot_result_histogram`, such as `bins` or `color`. `surface` : bool or dict If enabled, save surface maps for matched `hemi-L`/`hemi-R` result files. Surface plotting requires `lh` and `rh` surface inputs, either at the top level or inside `surface`. Optional keys include `medial_wall`, `color_range` and `cmap`. `plot_dir` : str or Path Custom figure directory. Defaults to `<result_folder>/figures`. `show` : bool Whether to display figures while saving. Defaults to False for pipeline use. `fail_on_error` : bool If False, plotting errors are reported as warnings and do not stop the statistical pipeline. Example:: plot_config = { "histogram": {"bins": 40}, "surface": { "enabled": True, "lh": lh, "rh": rh, "medial_wall": medial_wall, "color_range": (-0.2, 0.6), "cmap": "Reds" }, "show": False } |

**Returns or outputs**

See the function output or saved pipeline records for returned objects and generated files.


## IDM Reliability

`fmriha.stat.idm`

Split-half individual-difference-matrix reliability estimation.


### `sl_IDM_generation_mmap`

**Signature**

```python
fmriha.stat.idm.sl_IDM_generation_mmap(mmap_list, sl)
```

**Purpose**

Compute one searchlight-level IDM vector from preloaded subject data.

**Parameters**

| Parameter | Default | Meaning |
| --- | --- | --- |
| `mmap_list` | `required` | List of subject matrices, each with shape `(n_time, n_feature)`. |
| `sl` | `required` | Searchlight indices for one local region. |

**Returns or outputs**

ndarray Upper-triangular part of the subject-by-subject IDM matrix.

### `idm_pipe`

**Signature**

```python
fmriha.stat.idm.idm_pipe(work_dir, sub_list, step, modality, structure_name, file_filter, time_split, sls, njobs=5, verbose=1, method='response', prepare_inputs=True, target_structure=None, seed_sls=None, target_sls=None, zscore=True, target_filter=None, suffix=None, dtype=np.float32, summary_statistic='mean', delete_split_files=False, json_path='prep_log.json', overwrite=False, log_num='00004')
```

**Purpose**

Estimate split-half IDM reliability from response or derived data.

**Parameters**

| Parameter | Default | Meaning |
| --- | --- | --- |
| `work_dir` | `required` | Root HA working directory. |
| `sub_list` | `required` | Subject folder names, such as `['sub-01', 'sub-02']`. |
| `step` | `required` | Pipeline step and source modality. For `method='response'` this is also the IDM input location. For `method='connectivity'` and `method='representational_geometry'`, split files are prepared under `step/connectivity` or `step/representational_geometry`. |
| `modality` | `required` | Data modality label, such as response or connectivity. |
| `structure_name` | `required` | One or more structure folders to process. |
| `file_filter` | `required` | Base BIDS-style filter used to locate source files. Prepared IDM split files use compact names with only `sub`, `ses` and `split` fields; if `ses` is present here, it is also used when locating split files. |
| `time_split` | `required` | Split strategy. This function compares the two saved halves. |
| `sls` | `required` | Searchlight definitions used to build IDM vectors from the final split input files. For `method='representational_geometry'` the same searchlights are also used to build RG features unless another workflow prepares those files beforehand. |
| `njobs` | `5` | Number of parallel workers. |
| `verbose` | `1` | Joblib verbosity level. |
| `method` | `'response'` | Data representation used for IDM reliability. |
| `prepare_inputs` | `True` | If True, create the required split files before computing reliability. This keeps IDM preparation independent from the ISC pipelines. |
| `target_structure` | `None` | Connectivity-specific options used when `method='connectivity'` and `prepare_inputs=True`. |
| `seed_sls` | `None` | Searchlights used to summarize seed-side data. |
| `target_sls` | `None` | Searchlights used to summarize target-side data. |
| `zscore` | `True` | Whether to z-score data before connectivity or statistical computation. |
| `target_filter` | `None` | Selection filters used to choose matching files or records. |
| `suffix` | `None` | Optional suffix appended to final IDM reliability output filenames. |
| `dtype` | `np.float32` | Reserved for interface consistency with related pipelines. |
| `summary_statistic` | `'mean'` | Summary statistic computed across the raw split-half reliability vector. The final output is a pickle file containing a dictionary with `raw` and the selected summary statistic. |
| `delete_split_files` | `False` | If True, remove split-half files generated by this IDM run after the final reliability pickle files are saved. |
| `json_path` | `'prep_log.json'` | JSON log path. |
| `overwrite` | `False` | Whether to overwrite existing JSON log entries. |
| `log_num` | `'00004'` | Progress log identifier. |

**Returns or outputs**

See the function output or saved pipeline records for returned objects and generated files.


## bsMVPC

`fmriha.stat.bsmvpc`

Between-subject multivariate pattern correlation analysis.


### `window_generation`

**Signature**

```python
fmriha.stat.bsmvpc.window_generation(sub_data, window_length, window_step)
```

**Purpose**

Generate sliding windows and their index ranges for one subject matrix.

**Parameters**

| Parameter | Default | Meaning |
| --- | --- | --- |
| `sub_data` | `required` | Subject response matrix with shape `(n_time, n_feature)`. |
| `window_length` | `required` | Number of time points per window. |
| `window_step` | `required` | Step size between adjacent windows. |

**Returns or outputs**

tuple `(windows, window_index)` where `windows` has shape `(n_window, window_length, n_feature)` and `window_index` stores the inclusive start/end indices of each window.

### `window_filter`

**Signature**

```python
fmriha.stat.bsmvpc.window_filter(target_wd_idx, all_wd_idx)
```

**Purpose**

Return indices of windows that do not overlap with the target window.

**Parameters**

| Parameter | Default | Meaning |
| --- | --- | --- |
| `target_wd_idx` | `required` | Index range of the target sliding window. |
| `all_wd_idx` | `required` | Index ranges for all candidate sliding windows. |

**Returns or outputs**

See the function output or saved pipeline records for returned objects and generated files.

### `bsmvpc_pipeline`

**Signature**

```python
fmriha.stat.bsmvpc.bsmvpc_pipeline(work_dir, sub_list, step, modality, structure_name, file_filter, window_length, window_step, sls, njobs=5, verbose=1, dtype=np.float32, summary_statistic='mean', json_path='prep_log.json', overwrite=False, log_num='00003', suffix=None, plot_config=None)
```

**Purpose**

Run bsMVPC on subject response data in the HA pipeline layout.

**Parameters**

| Parameter | Default | Meaning |
| --- | --- | --- |
| `work_dir` | `required` | Root HA working directory. |
| `sub_list` | `required` | Subject folder names, such as `['sub-01', 'sub-02']`. |
| `step` | `required` | Pipeline step and modality used to locate the input files. |
| `modality` | `required` | Data modality label, such as response or connectivity. |
| `structure_name` | `required` | One or more structure folders to process. |
| `file_filter` | `required` | BIDS-style filter passed to `fio.bids_filter`. |
| `window_length` | `required` | Number of time points per classification window. |
| `window_step` | `required` | Step size between adjacent windows. |
| `sls` | `required` | Searchlight definitions. |
| `njobs` | `5` | Number of parallel workers. |
| `verbose` | `1` | Joblib verbosity level. |
| `dtype` | `np.float32` | Output dtype for stored bsMVPC values. |
| `summary_statistic` | `'mean'` | Summary statistic saved alongside the raw bsMVPC matrix. |
| `json_path` | `'prep_log.json'` | JSON log path. |
| `overwrite` | `False` | Whether to overwrite existing JSON log entries. |
| `log_num` | `'00003'` | Progress log identifier. |
| `suffix` | `None` | Optional suffix appended to the output filename. |
| `plot_config` | `None` | Optional plotting configuration used after result pkl files are saved. `None` or `False` disables plotting and keeps the original pipeline behavior. `True` saves default histogram figures only. A dictionary gives explicit control. Common keys are: `method` : str Result field to plot, usually `'mean'` or `'median'`. Defaults to `summary_statistic`. `histogram` : bool or dict If True, save histogram figures to `figures` under the result folder. A dict may include arguments for `fmriha.plot.plot_result_histogram`, such as `bins` or `color`. `surface` : bool or dict If enabled, save surface maps for matched `hemi-L`/`hemi-R` result files. Surface plotting requires `lh` and `rh` surface inputs, either at the top level or inside `surface`. Optional keys include `medial_wall`, `color_range` and `cmap`. `plot_dir` : str or Path Custom figure directory. Defaults to `<result_folder>/figures`. `show` : bool Whether to display figures while saving. Defaults to False for pipeline use. `fail_on_error` : bool If False, plotting errors are reported as warnings and do not stop the statistical pipeline. Example:: plot_config = { "histogram": {"bins": 40}, "surface": { "enabled": True, "lh": lh, "rh": rh, "medial_wall": medial_wall, "color_range": (0.0, 0.8), "cmap": "Reds" }, "show": False } |

**Returns or outputs**

See the function output or saved pipeline records for returned objects and generated files.


## PSF

`fmriha.stat.psf`

Within-subject and between-subject point-spread-function analysis.


### `fisher_rmean`

**Signature**

```python
fmriha.stat.psf.fisher_rmean(rvals, axis=None)
```

**Purpose**

Compute the Fisher z mean of correlation values.

**Parameters**

| Parameter | Default | Meaning |
| --- | --- | --- |
| `rvals` | `required` | Correlation values. |
| `axis` | `None` | Axis along which the mean is computed. |

**Returns or outputs**

ndarray Fisher z averaged correlations.

### `bspsf_pipeline`

**Signature**

```python
fmriha.stat.psf.bspsf_pipeline(work_dir, sub_list, step, modality, structure_name, file_filter, sls, njobs=5, verbose=1, dtype=np.float32, summary_statistic='mean', json_path='prep_log.json', overwrite=False, log_num='00004', suffix=None)
```

**Purpose**

Run between-subject PSF using annular searchlights.

**Parameters**

| Parameter | Default | Meaning |
| --- | --- | --- |
| `work_dir` | `required` | Root working directory. |
| `sub_list` | `required` | Subject identifiers. |
| `step` | `required` | Processing step folder containing source data. |
| `modality` | `required` | Modality folder containing source data. |
| `structure_name` | `required` | Structure name(s), such as `'hemi-L'` or `'volume-hippocampus'`. |
| `file_filter` | `required` | BIDS-style filters used to locate exactly one source file per subject. |
| `sls` | `required` | Annular searchlights. The expected format is one ring per center. |
| `njobs` | `5` | Number of parallel workers. |
| `verbose` | `1` | Joblib verbosity level. |
| `dtype` | `np.float32` | Output precision. |
| `summary_statistic` | `'mean'` | Group summary statistic saved alongside the raw results. |
| `json_path` | `'prep_log.json'` | JSON log path. |
| `overwrite` | `False` | Whether to overwrite the JSON log entry target. |
| `log_num` | `'00004'` | Progress log identifier. |
| `suffix` | `None` | Optional filename suffix for saved outputs. |

**Returns or outputs**

See the function output or saved pipeline records for returned objects and generated files.

### `wspsf_pipeline`

**Signature**

```python
fmriha.stat.psf.wspsf_pipeline(work_dir, sub_list, step, modality, structure_name, file_filter, sls, njobs=5, verbose=1, dtype=np.float32, summary_statistic='mean', json_path='prep_log.json', overwrite=False, log_num='00005', suffix=None)
```

**Purpose**

Run within-subject PSF using annular searchlights.

**Parameters**

| Parameter | Default | Meaning |
| --- | --- | --- |
| `work_dir` | `required` | Root working directory. |
| `sub_list` | `required` | Subject identifiers. |
| `step` | `required` | Processing step folder containing source data. |
| `modality` | `required` | Modality folder containing source data. |
| `structure_name` | `required` | Structure name(s), such as `'hemi-L'` or `'volume-hippocampus'`. |
| `file_filter` | `required` | BIDS-style filters used to locate exactly one source file per subject. |
| `sls` | `required` | Annular searchlights. The expected format is one ring per center. |
| `njobs` | `5` | Number of parallel workers. |
| `verbose` | `1` | Joblib verbosity level. |
| `dtype` | `np.float32` | Output precision. |
| `summary_statistic` | `'mean'` | Group summary statistic saved alongside the raw results. |
| `json_path` | `'prep_log.json'` | JSON log path. |
| `overwrite` | `False` | Whether to overwrite the JSON log entry target. |
| `log_num` | `'00005'` | Progress log identifier. |
| `suffix` | `None` | Optional filename suffix for saved outputs. |

**Returns or outputs**

See the function output or saved pipeline records for returned objects and generated files.


## MVPA Basics

`fmriha.stat.mvpa.basic`

SVM classification, permutation testing, and searchlight-wise MVPA helpers.


### `fit_svm`

**Signature**

```python
fmriha.stat.mvpa.basic.fit_svm(train_feature, train_label, C=1.0, kernel='linear')
```

**Purpose**

Fit an SVM model for classification.

**Parameters**

| Parameter | Default | Meaning |
| --- | --- | --- |
| `train_feature` | `required` | Training feature matrix with shape `(n_sample, n_feature)`. |
| `train_label` | `required` | Training label vector with shape `(n_sample,)`. |
| `C` | `1.0` | SVM regularization parameter. |
| `kernel` | `'linear'` | Kernel type. `linear_fast` uses `LinearSVC`. |

**Returns or outputs**

sklearn estimator Fitted SVM model.

### `predict_svm`

**Signature**

```python
fmriha.stat.mvpa.basic.predict_svm(model, test_feature)
```

**Purpose**

Predict labels and decision values from a fitted SVM model.

**Parameters**

| Parameter | Default | Meaning |
| --- | --- | --- |
| `model` | `required` | Trained SVM model returned by :func:`fit_svm`. |
| `test_feature` | `required` | Test feature matrix with shape `(n_sample, n_feature)`. |

**Returns or outputs**

tuple of ndarray Predicted labels and score/probability matrix.

### `classification_evaluation`

**Signature**

```python
fmriha.stat.mvpa.basic.classification_evaluation(test_label, test_label_pred, test_label_proba=None, classes_order=None)
```

**Purpose**

Evaluate classification results with accuracy and optional AUC.

**Parameters**

| Parameter | Default | Meaning |
| --- | --- | --- |
| `test_label` | `required` | True label vector. |
| `test_label_pred` | `required` | Predicted label vector. |
| `test_label_proba` | `None` | Probability matrix or decision-function output. |
| `classes_order` | `None` | Class order corresponding to the columns of `test_label_proba`. |

**Returns or outputs**

dict Dictionary with at least `accuracy` and, when possible, `auc`.

### `normalize_basic`

**Signature**

```python
fmriha.stat.mvpa.basic.normalize_basic(train_feature, test_feature, return_scaler=False)
```

**Purpose**

Standardize train and test features using the training-set statistics.

**Parameters**

| Parameter | Default | Meaning |
| --- | --- | --- |
| `train_feature` | `required` | Training feature matrix with shape `(n_sample, n_feature)`. |
| `test_feature` | `required` | Test feature matrix with shape `(n_sample, n_feature)`. |
| `return_scaler` | `False` | Whether to also return the fitted `StandardScaler`. |

**Returns or outputs**

tuple Normalized train/test matrices, optionally followed by the scaler.

### `cv_common`

**Signature**

```python
fmriha.stat.mvpa.basic.cv_common(feature_all, label_all, model_fn, eval_fn, cv, cv_strategy)
```

**Purpose**

Run the shared cross-validation loop used by SVM classification.

**Parameters**

| Parameter | Default | Meaning |
| --- | --- | --- |
| `feature_all` | `required` | Feature matrix with shape `(n_sample, n_feature)`. |
| `label_all` | `required` | Label vector with shape `(n_sample,)`. |
| `model_fn` | `required` | Function that fits and returns a classifier from train data. |
| `eval_fn` | `required` | Function that evaluates predictions on the test split. |
| `cv` | `required` | Cross-validation splitter object. |
| `cv_strategy` | `required` | Strategy name. Kept for compatibility and error context. |

**Returns or outputs**

dict Mean metric values across folds.

### `run_svm_classification`

**Signature**

```python
fmriha.stat.mvpa.basic.run_svm_classification(feature_all, label_all, cv_strategy='StratifiedKFold', C=1.0, kernel='linear', n_splits=5, random_seed=42)
```

**Purpose**

Run cross-validated SVM classification.

**Parameters**

| Parameter | Default | Meaning |
| --- | --- | --- |
| `feature_all` | `required` | Feature matrix with shape `(n_sample, n_feature)`. |
| `label_all` | `required` | Label vector with shape `(n_sample,)`. |
| `cv_strategy` | `'StratifiedKFold'` | Cross-validation strategy. |
| `C` | `1.0` | SVM regularization parameter. |
| `kernel` | `'linear'` | Kernel type. |
| `n_splits` | `5` | Number of folds for K-fold based CV. |
| `random_seed` | `42` | Random seed for shuffled splitters. |

**Returns or outputs**

dict Mean metric values across folds.

### `svm_permutation`

**Signature**

```python
fmriha.stat.mvpa.basic.svm_permutation(feature_all, label_all, cv_strategy='StratifiedKFold', C=1.0, kernel='linear', n_splits=3, random_seed=42, perm_time=10, evaluate='accuracy', n_jobs=5)
```

**Purpose**

Run permutation testing for cross-validated SVM classification.

**Parameters**

| Parameter | Default | Meaning |
| --- | --- | --- |
| `feature_all` | `required` | Feature matrix with shape `(n_sample, n_feature)`. |
| `label_all` | `required` | Label vector with shape `(n_sample,)`. |
| `cv_strategy` | `'StratifiedKFold'` | Cross-validation strategy. |
| `C` | `1.0` | SVM regularization parameter. |
| `kernel` | `'linear'` | Kernel type. |
| `n_splits` | `3` | Number of folds for K-fold based CV. |
| `random_seed` | `42` | Random seed used for both CV and label permutation. |
| `perm_time` | `10` | Number of permutations. |
| `evaluate` | `'accuracy'` | Primary metric name retained for compatibility. |
| `n_jobs` | `5` | Number of parallel jobs. Use `None` to run permutations serially. |

**Returns or outputs**

dict Real metrics, null distributions, and p-values.

### `load_gifti_data`

**Signature**

```python
fmriha.stat.mvpa.basic.load_gifti_data(gifti_file)
```

**Purpose**

Load data from a GIFTI file.

**Parameters**

| Parameter | Default | Meaning |
| --- | --- | --- |
| `gifti_file` | `required` | Path to the GIFTI file. |

**Returns or outputs**

tuple `(gii_data, nt, nv)` where `gii_data` has shape `(time_points, n_vertices)`.

### `load_cifti_data`

**Signature**

```python
fmriha.stat.mvpa.basic.load_cifti_data(cifti_file, surface=True, medial_wall=True, sub_cortical=False)
```

**Purpose**

Load selected data components from a CIFTI file.

**Parameters**

| Parameter | Default | Meaning |
| --- | --- | --- |
| `cifti_file` | `required` | Path to the CIFTI file. |
| `surface` | `True` | Whether to return left/right cortical surface data. |
| `medial_wall` | `True` | Whether to return the corresponding medial-wall masks. |
| `sub_cortical` | `False` | Whether to return stacked sub-cortical data. |

**Returns or outputs**

list Ordered list containing the requested outputs.

### `run_svm_classification_sls`

**Signature**

```python
fmriha.stat.mvpa.basic.run_svm_classification_sls(feature_all, label_all, sls, mask=None, cv_strategy='StratifiedKFold', C=1.0, kernel='linear', n_splits=3, random_seed=42, n_jobs=5, evaluate='accuracy')
```

**Purpose**

Run searchlight-wise cross-validated SVM classification.

**Parameters**

| Parameter | Default | Meaning |
| --- | --- | --- |
| `feature_all` | `required` | Feature matrix with shape `(n_sample, n_feature)`. |
| `label_all` | `required` | Label vector with shape `(n_sample,)`. |
| `sls` | `required` | Searchlight index arrays. |
| `mask` | `None` | Reserved for future use. |
| `cv_strategy` | `'StratifiedKFold'` | Cross-validation strategy. |
| `C` | `1.0` | SVM regularization parameter. |
| `kernel` | `'linear'` | Kernel type. |
| `n_splits` | `3` | Number of folds for K-fold based CV. |
| `random_seed` | `42` | Random seed for shuffled CV. |
| `n_jobs` | `5` | Number of parallel jobs across searchlights. |
| `evaluate` | `'accuracy'` | Primary metric name to expose in the output dictionary. |

**Returns or outputs**

dict Searchlight result maps.

### `run_svm_permutation_sls`

**Signature**

```python
fmriha.stat.mvpa.basic.run_svm_permutation_sls(feature_all, label_all, sls, cv_strategy='StratifiedKFold', C=1.0, kernel='linear', n_splits=3, random_seed=42, perm_time=10, evaluate='accuracy', n_jobs=5)
```

**Purpose**

Run searchlight-wise SVM permutation testing.

**Parameters**

| Parameter | Default | Meaning |
| --- | --- | --- |
| `feature_all` | `required` | Feature matrix with shape `(n_sample, n_feature)`. |
| `label_all` | `required` | Label vector with shape `(n_sample,)`. |
| `sls` | `required` | Searchlight index arrays. |
| `cv_strategy` | `'StratifiedKFold'` | Cross-validation strategy. |
| `C` | `1.0` | SVM regularization parameter. |
| `kernel` | `'linear'` | Kernel type. |
| `n_splits` | `3` | Number of folds for K-fold based CV. |
| `random_seed` | `42` | Random seed for shuffled CV and permutations. |
| `perm_time` | `10` | Number of permutations. |
| `evaluate` | `'accuracy'` | Metric name used to form the output maps. |
| `n_jobs` | `5` | Number of parallel jobs across searchlights. |

**Returns or outputs**

tuple of ndarray Real-value map, p-value map, and AUC map.

### `svm_pipeline`

**Signature**

```python
fmriha.stat.mvpa.basic.svm_pipeline(feature_file_list, label_all, sls, analysis='cv', mapper=None, file_type='gifti', mask=None, cv_strategy='StratifiedKFold', C=1.0, kernel='linear', n_splits=3, random_seed=42, n_jobs=5, evaluate='accuracy', perm_time=10)
```

**Purpose**

Load feature files and run MVPA analysis for GIFTI or CIFTI data.

**Parameters**

| Parameter | Default | Meaning |
| --- | --- | --- |
| `feature_file_list` | `required` | Input feature files, one file per sample. |
| `label_all` | `required` | Label vector for all samples. |
| `sls` | `required` | Searchlight definitions. For CIFTI, use a dict with keys `'l'` and `'r'`. |
| `analysis` | `'cv'` | Analysis type. |
| `mapper` | `None` | Optional feature mapper. For CIFTI, this should be a dict by hemisphere. |
| `file_type` | `'gifti'` | Input file type. |
| `mask` | `None` | Reserved for future use. |
| `cv_strategy` | `'StratifiedKFold'` | Cross-validation strategy. |
| `C` | `1.0` | SVM regularization parameter. |
| `kernel` | `'linear'` | Kernel type. |
| `n_splits` | `3` | Number of folds for K-fold based CV. |
| `random_seed` | `42` | Random seed. |
| `n_jobs` | `5` | Number of parallel jobs. |
| `evaluate` | `'accuracy'` | Metric name used in searchlight outputs. |
| `perm_time` | `10` | Number of permutations. |

**Returns or outputs**

object Analysis results for the requested file type and analysis mode.


## RSA

`fmriha.stat.mvpa.RSA_core`

Representational similarity analysis and FDR correction functions.


### `safe_z`

**Signature**

```python
fmriha.stat.mvpa.RSA_core.safe_z(x, axis=0, ddof=1)
```

**Purpose**

Z-score an array while protecting against near-zero variance.

**Parameters**

| Parameter | Default | Meaning |
| --- | --- | --- |
| `x` | `required` | Input array. |
| `axis` | `0` | Axis along which z-scoring is applied. |
| `ddof` | `1` | Delta degrees of freedom used in standard deviation. |

**Returns or outputs**

ndarray Z-scored array.

### `sim_matrix`

**Signature**

```python
fmriha.stat.mvpa.RSA_core.sim_matrix(beta_mat, method='correlation', output='matrix')
```

**Purpose**

Compute a similarity matrix from beta-pattern data.

**Parameters**

| Parameter | Default | Meaning |
| --- | --- | --- |
| `beta_mat` | `required` | Pattern matrix with shape `(n_condition, n_feature)`. |
| `method` | `'correlation'` | Similarity metric. |
| `output` | `'matrix'` | Return the full similarity matrix or its upper triangle. |

**Returns or outputs**

ndarray Similarity matrix or upper-triangle vector.

### `rsa_single`

**Signature**

```python
fmriha.stat.mvpa.RSA_core.rsa_single(reference_rdm_mat, beta_mat, method='correlation', output='matrix')
```

**Purpose**

Compute one RSA correlation between a reference RDM and brain patterns.

**Parameters**

| Parameter | Default | Meaning |
| --- | --- | --- |
| `reference_rdm_mat` | `required` | Reference RDM as a square matrix or upper-triangle vector. |
| `beta_mat` | `required` | Brain beta-pattern matrix with shape `(n_condition, n_feature)`. |
| `method` | `'correlation'` | Similarity metric used to build the brain similarity matrix. |
| `output` | `'matrix'` | Intermediate output mode used when building the brain RDM. |

**Returns or outputs**

float RSA correlation value.

### `rsa_permutation_single`

**Signature**

```python
fmriha.stat.mvpa.RSA_core.rsa_permutation_single(reference_rdm_mat, beta_mat, method='correlation', n_perm=1000)
```

**Purpose**

Compute one permutation-based RSA test using the Mantel test.

**Parameters**

| Parameter | Default | Meaning |
| --- | --- | --- |
| `reference_rdm_mat` | `required` | Reference RDM matrix. |
| `beta_mat` | `required` | Brain beta-pattern matrix with shape `(n_condition, n_feature)`. |
| `method` | `'correlation'` | Similarity metric used to build the brain similarity matrix. |
| `n_perm` | `1000` | Number of permutations passed to the Mantel test. |

**Returns or outputs**

tuple `(r, p, n)` from `skbio.stats.distance.mantel`.

### `rsa_rval_sls`

**Signature**

```python
fmriha.stat.mvpa.RSA_core.rsa_rval_sls(reference_rdm_mat, beta_mat, sls, method='correlation', output='matrix', n_jobs=10)
```

**Purpose**

Compute RSA r-values for all searchlights.

**Parameters**

| Parameter | Default | Meaning |
| --- | --- | --- |
| `reference_rdm_mat` | `required` | Reference RDM as a square matrix or upper-triangle vector. |
| `beta_mat` | `required` | Brain beta-pattern matrix with shape `(n_condition, n_feature)`. |
| `sls` | `required` | Searchlight index arrays. |
| `method` | `'correlation'` | Similarity metric. |
| `output` | `'matrix'` | Intermediate output mode used when building searchlight RDMs. |
| `n_jobs` | `10` | Number of parallel jobs. |

**Returns or outputs**

ndarray RSA r-values for all searchlights.

### `rsa_permutation_sls`

**Signature**

```python
fmriha.stat.mvpa.RSA_core.rsa_permutation_sls(reference_rdm_mat, beta_mat, sls, method='correlation', n_perm=1000, n_jobs=10)
```

**Purpose**

Compute permutation-based RSA for all searchlights.

**Parameters**

| Parameter | Default | Meaning |
| --- | --- | --- |
| `reference_rdm_mat` | `required` | Reference RDM matrix. |
| `beta_mat` | `required` | Brain beta-pattern matrix with shape `(n_condition, n_feature)`. |
| `sls` | `required` | Searchlight index arrays. |
| `method` | `'correlation'` | Similarity metric. |
| `n_perm` | `1000` | Number of permutations for the Mantel test. |
| `n_jobs` | `10` | Number of parallel jobs. |

**Returns or outputs**

tuple of ndarray Arrays of Mantel `r`, `p`, and `n` values across searchlights.

### `fdr_correction`

**Signature**

```python
fmriha.stat.mvpa.RSA_core.fdr_correction(p_vals, alpha=0.05, method='bh')
```

**Purpose**

Apply Benjamini-Hochberg or Benjamini-Yekutieli FDR correction.

**Parameters**

| Parameter | Default | Meaning |
| --- | --- | --- |
| `p_vals` | `required` | One-dimensional array of p-values. |
| `alpha` | `0.05` | Significance level. |
| `method` | `'bh'` | FDR correction method. |

**Returns or outputs**

tuple Adjusted p-values and rejection mask.


## Nested-CV MVPA

`fmriha.stat.mvpa.cv_nested`

Nested cross-validation utilities for searchlight-wise SVM analyses.


### `make_splitter`

**Signature**

```python
fmriha.stat.mvpa.cv_nested.make_splitter(kind, n_splits=None, shuffle=True, random_state=None)
```

**Purpose**

Build a scikit-learn CV splitter from a configuration name.

**Parameters**

| Parameter | Default | Meaning |
| --- | --- | --- |
| `kind` | `required` | Cross-validation splitter name. |
| `n_splits` | `None` | Number of folds for K-fold style splitters. |
| `shuffle` | `True` | Whether to shuffle samples before splitting. |
| `random_state` | `None` | Random seed used when `shuffle=True`. |

**Returns or outputs**

tuple `(splitter, needs)` where `needs` describes whether `y` or `groups` are required by the splitter logic.

### `class NoCV`

**Purpose**

Legacy splitter that returns one fold with an empty test set.

**Public methods**

### `NoCV.split`

**Signature**

```python
fmriha.stat.mvpa.cv_nested.NoCV.split(X, y=None, groups=None)
```

**Purpose**

Performs the operation indicated by the function name and arguments.

**Parameters**

| Parameter | Default | Meaning |
| --- | --- | --- |
| `X` | `required` | Input data matrix or point cloud. |
| `y` | `None` | Optional target labels used by scikit-learn splitter interfaces. |
| `groups` | `None` | Optional group labels used by scikit-learn splitter interfaces. |

**Returns or outputs**

See the function output or saved pipeline records for returned objects and generated files.

### `NoCV.get_n_splits`

**Signature**

```python
fmriha.stat.mvpa.cv_nested.NoCV.get_n_splits(X=None, y=None, groups=None)
```

**Purpose**

Performs the operation indicated by the function name and arguments.

**Parameters**

| Parameter | Default | Meaning |
| --- | --- | --- |
| `X` | `None` | Input data matrix or point cloud. |
| `y` | `None` | Optional target labels used by scikit-learn splitter interfaces. |
| `groups` | `None` | Optional group labels used by scikit-learn splitter interfaces. |

**Returns or outputs**

See the function output or saved pipeline records for returned objects and generated files.

### `class SafeNoCV`

**Purpose**

A no-cross-validation splitter with a non-empty test set.

**Public methods**

### `SafeNoCV.split`

**Signature**

```python
fmriha.stat.mvpa.cv_nested.SafeNoCV.split(X, y=None, groups=None)
```

**Purpose**

Performs the operation indicated by the function name and arguments.

**Parameters**

| Parameter | Default | Meaning |
| --- | --- | --- |
| `X` | `required` | Input data matrix or point cloud. |
| `y` | `None` | Optional target labels used by scikit-learn splitter interfaces. |
| `groups` | `None` | Optional group labels used by scikit-learn splitter interfaces. |

**Returns or outputs**

See the function output or saved pipeline records for returned objects and generated files.

### `SafeNoCV.get_n_splits`

**Signature**

```python
fmriha.stat.mvpa.cv_nested.SafeNoCV.get_n_splits(X=None, y=None, groups=None)
```

**Purpose**

Performs the operation indicated by the function name and arguments.

**Parameters**

| Parameter | Default | Meaning |
| --- | --- | --- |
| `X` | `None` | Input data matrix or point cloud. |
| `y` | `None` | Optional target labels used by scikit-learn splitter interfaces. |
| `groups` | `None` | Optional group labels used by scikit-learn splitter interfaces. |

**Returns or outputs**

See the function output or saved pipeline records for returned objects and generated files.

### `nested_cv_svm`

**Signature**

```python
fmriha.stat.mvpa.cv_nested.nested_cv_svm(input_df, outer_cv, inner_cv, sls, param_grid_C=(0.01, 0.1, 1, 10), evaluate='accuracy', permute_labels=False, rng=None, random_seed=42, n_jobs=25, verbose=0)
```

**Purpose**

Run nested cross-validated SVM classification for each searchlight.

**Parameters**

| Parameter | Default | Meaning |
| --- | --- | --- |
| `input_df` | `required` | Input table containing at least `feature` and `label` columns. |
| `outer_cv` | `required` | CV configuration dictionaries. |
| `inner_cv` | `required` | Inner cross-validation configuration used for hyperparameter selection. |
| `sls` | `required` | Searchlight index arrays. |
| `param_grid_C` | `(0.01, 0.1, 1, 10)` | Candidate `C` values used in the inner-loop model selection. |
| `evaluate` | `'accuracy'` | Metric used to select the best hyperparameter and report outputs. |
| `permute_labels` | `False` | Whether to shuffle training labels within each outer fold. |
| `rng` | `None` | Optional RNG used only to derive a deterministic permutation seed base. |
| `random_seed` | `42` | Base random seed. |
| `n_jobs` | `25` | Number of parallel jobs across searchlights. |
| `verbose` | `0` | Joblib verbosity level. |

**Returns or outputs**

dict Mean metric maps across searchlights.

### `nested_cv_svm_permute`

**Signature**

```python
fmriha.stat.mvpa.cv_nested.nested_cv_svm_permute(input_df, outer_cv, inner_cv, sls, file_type, param_grid_C=(1, 10), evaluate='accuracy', random_seed=42, n_jobs=25, n_perm=1000)
```

**Purpose**

Run a permutation test for the basic nested-CV SVM workflow.

**Parameters**

| Parameter | Default | Meaning |
| --- | --- | --- |
| `input_df` | `required` | Input table containing at least `feature` and `label` columns. |
| `outer_cv` | `required` | CV configuration dictionaries. |
| `inner_cv` | `required` | Inner cross-validation configuration used for hyperparameter selection. |
| `sls` | `required` | Searchlight index arrays. |
| `file_type` | `required` | Kept for backward-compatible API parity. Not used in this function. |
| `param_grid_C` | `(1, 10)` | Candidate `C` values. |
| `evaluate` | `'accuracy'` | Metric used for significance testing. |
| `random_seed` | `42` | Base random seed. |
| `n_jobs` | `25` | Number of parallel jobs across searchlights. |
| `n_perm` | `1000` | Number of permutations. |

**Returns or outputs**

tuple Observed results, null distribution, and empirical p-values.

### `leave_sub_run_out`

**Signature**

```python
fmriha.stat.mvpa.cv_nested.leave_sub_run_out(input_df, outer_cv, inner_cv, sls, param_grid_C=(1, 10), evaluate='accuracy', random_seed=42, n_jobs=25, verbose=0)
```

**Purpose**

Run nested CV with subject-level and run-level splitting.

**Parameters**

| Parameter | Default | Meaning |
| --- | --- | --- |
| `input_df` | `required` | Input table containing `feature`, `label`, `subject_id`, and `run_id`. |
| `outer_cv` | `required` | CV configuration dictionaries. |
| `inner_cv` | `required` | Inner cross-validation configuration used for hyperparameter selection. |
| `sls` | `required` | Searchlight index arrays. |
| `param_grid_C` | `(1, 10)` | Candidate `C` values used for inner-loop model selection. |
| `evaluate` | `'accuracy'` | Metric used for hyperparameter selection and output summaries. |
| `random_seed` | `42` | Base random seed used by splitters. |
| `n_jobs` | `25` | Number of parallel jobs across searchlights. |
| `verbose` | `0` | Joblib verbosity level. |

**Returns or outputs**

dict Mean metric maps and normalized confusion matrices across searchlights.

### `leave_sub_run_out_permute`

**Signature**

```python
fmriha.stat.mvpa.cv_nested.leave_sub_run_out_permute(input_df, outer_cv, inner_cv, sls, param_grid_C=(1, 10), evaluate='accuracy', random_seed=42, n_jobs=25, n_perm=1000, permute_labels='subject')
```

**Purpose**

Run a permutation test for subject/run-aware nested CV.

**Parameters**

| Parameter | Default | Meaning |
| --- | --- | --- |
| `input_df` | `required` | Input table containing `feature`, `label`, `subject_id`, and `run_id`. |
| `outer_cv` | `required` | CV configuration dictionaries. |
| `inner_cv` | `required` | Inner cross-validation configuration used for hyperparameter selection. |
| `sls` | `required` | Searchlight index arrays. |
| `param_grid_C` | `(1, 10)` | Candidate `C` values. |
| `evaluate` | `'accuracy'` | Metric used for significance testing. |
| `random_seed` | `42` | Base random seed. |
| `n_jobs` | `25` | Number of parallel jobs across searchlights. |
| `n_perm` | `1000` | Number of permutations. |
| `permute_labels` | `'subject'` | Permutation level. |

**Returns or outputs**

tuple Observed results, null distribution, and empirical p-values.


## Plotting

`fmriha.plot.plot_basic`

Plotting helpers for histogram and surface-map summaries.


### `plot_result_histogram`

**Signature**

```python
fmriha.plot.plot_basic.plot_result_histogram(result_files, method='mean', metric=None, title=None, output=None, show=True, bins=30, color='#4C72B0')
```

**Purpose**

Plot a histogram from ISC or bsMVPC pipe output pkl files.

**Parameters**

| Parameter | Default | Meaning |
| --- | --- | --- |
| `result_files` | `required` | One result pkl file, multiple result pkl files, or a mapping from structure names to result pkl files. A single `hemi-L` or `hemi-R` path automatically picks up the matching opposite hemisphere file when it exists in the same folder. |
| `method` | `'mean'` | Result field to plot, such as `'mean'`, `'median'`, or `'raw'`. `metric` is accepted as an alias for `method`. |
| `metric` | `None` | Similarity or distance metric. |
| `title` | `None` | Plot title. If None, a title is inferred from the selected structures. |
| `output` | `None` | Save path. If None, the figure is displayed and not saved. |
| `show` | `True` | When `output` is provided, show the figure after saving if True. When `output` is None, the figure is always displayed. |
| `bins` | `30` | Histogram bin specification passed to `matplotlib`. |
| `color` | `'#4C72B0'` | Histogram bar color. |

**Returns or outputs**

fig, ax : tuple Matplotlib figure and axes.

### `plot_surface_result_map`

**Signature**

```python
fmriha.plot.plot_basic.plot_surface_result_map(result_files, lh, rh, method='mean', metric=None, medial_wall=None, color_range=None, cmap='Reds', title=None, output=None, show=True, zoom=1.2, size=(800, 250), layout='row')
```

**Purpose**

Plot a surface result map from ISC or bsMVPC pipe output pkl files.

**Parameters**

| Parameter | Default | Meaning |
| --- | --- | --- |
| `result_files` | `required` | One or more result pkl files. A single hemi pkl path automatically loads the opposite hemi file when present. |
| `lh` | `required` | Left and right surface inputs accepted by `surfplot.Plot`. |
| `rh` | `required` | Right-hemisphere surface input or right-hemisphere data container. |
| `method` | `'mean'` | Result field to draw, such as `'mean'`, `'median'`, or `'raw'`. |
| `metric` | `None` | Similarity or distance metric. |
| `medial_wall` | `None` | Medial wall masks keyed by `'l'` and `'r'`. When provided, cortical data are patched back into full surface vertex arrays. |
| `color_range` | `None` | Color range passed to `Plot.add_layer`. |
| `cmap` | `'Reds'` | Matplotlib colormap name. |
| `title` | `None` | Title prefix. Hemisphere means are appended automatically. |
| `output` | `None` | Save path. If None, the figure is displayed and not saved. |
| `show` | `True` | When `output` is provided, show the figure after saving if True. When `output` is None, the figure is always displayed. |
| `zoom` | `1.2` | Passed to `surfplot.Plot`. |
| `size` | `(800, 250)` | Figure size passed to the plotting backend. |
| `layout` | `'row'` | Surface-plot layout passed to the plotting backend. |

**Returns or outputs**

fig Surface plot figure.

### `plot_histogram`

**Signature**

```python
fmriha.plot.plot_basic.plot_histogram(hemisphere_data, method='mean', title=None, output=None, show=True, bins=30, **kwargs)
```

**Purpose**

Backward-compatible wrapper for :func:`plot_result_histogram`.

**Parameters**

| Parameter | Default | Meaning |
| --- | --- | --- |
| `hemisphere_data` | `required` | Hemisphere-keyed result data or files to plot. |
| `method` | `'mean'` | Analysis method or similarity metric requested by the function. |
| `title` | `None` | Optional figure title. |
| `output` | `None` | Optional path where the figure is saved. |
| `show` | `True` | Whether to display the figure after plotting or saving. |
| `bins` | `30` | Histogram bin specification. |
| `**kwargs` | `optional` | Additional keyword arguments passed to the underlying routine. |

**Returns or outputs**

See the function output or saved pipeline records for returned objects and generated files.

### `plot_surf_results`

**Signature**

```python
fmriha.plot.plot_basic.plot_surf_results(hemisphere_data, medial_wall=None, lh=None, rh=None, color_range=None, title=None, output=None, show=True, method='mean', cmap='Reds', **kwargs)
```

**Purpose**

Backward-compatible wrapper for :func:`plot_surface_result_map`.

**Parameters**

| Parameter | Default | Meaning |
| --- | --- | --- |
| `hemisphere_data` | `required` | Hemisphere-keyed result data or files to plot. |
| `medial_wall` | `None` | Mask or indices identifying medial-wall vertices to remove or reinsert. |
| `lh` | `None` | Left-hemisphere surface input or left-hemisphere data container. |
| `rh` | `None` | Right-hemisphere surface input or right-hemisphere data container. |
| `color_range` | `None` | Value range used for surface-map coloring. |
| `title` | `None` | Optional figure title. |
| `output` | `None` | Optional path where the figure is saved. |
| `show` | `True` | Whether to display the figure after plotting or saving. |
| `method` | `'mean'` | Analysis method or similarity metric requested by the function. |
| `cmap` | `'Reds'` | Matplotlib colormap name. |
| `**kwargs` | `optional` | Additional keyword arguments passed to the underlying routine. |

**Returns or outputs**

See the function output or saved pipeline records for returned objects and generated files.
